In [2]:
# ================================================================================
# BLOQUE 1: IMPORTACIONES Y CONFIGURACIÓN
# ================================================================================

# --- Librerías Estándar de Python ---
import os
import time
import warnings
from dateutil.relativedelta import relativedelta

# --- Librerías de Terceros (Análisis y Datos) ---
import numpy as np
import pandas as pd
import pandas_datareader.data as web
import datetime
import requests
from scipy.stats import spearmanr
from scipy.stats.mstats import winsorize

# --- Librerías de Visualización ---
import matplotlib.pyplot as plt

# --- Librerías de Machine Learning (Modelos) ---
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNetCV, LinearRegression
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

# --- Librerías de Machine Learning (Utilidades y Métricas) ---
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, KFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ================================================================================
# --- CONFIGURACIÓN GENERAL DEL SCRIPT ---
# ================================================================================

# Ignorar warnings para una salida más limpia durante la ejecución
warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None     

In [3]:
# ===============================================================================
# BLOQUE 2 · Configuración de la API Finnhub (segura)
# ===============================================================================

# Cargar variables desde .env si existe (requiere python-dotenv)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

FINNHUB_API_KEY = os.getenv("FINNHUB_API_KEY", "").strip()

if not FINNHUB_API_KEY:
    print("⚠️  FINNHUB_API_KEY no está definida en el entorno.")
    ENABLE_FINNHUB = False
else:
    ENABLE_FINNHUB = True
    print("🔐 FINNHUB_API_KEY detectada (OK).")

CACHE_FILE    = "finnhub_sector_industry.pkl"  # Cache local para no repetir descargas
RATE_LIMIT    = 60  # Límite de llamadas por minuto para el plan gratuito de Finnhub
SLEEP_SECONDS = 61  # Pausa tras RATE_LIMIT llamadas (un segundo extra por seguridad)


🔐 FINNHUB_API_KEY detectada (OK).


In [4]:
# ===============================================================================
# BLOQUE 3· Reproducibilidad total (semillas, env, logging, metadatos)
# ===============================================================================

import os, sys, json, random, platform, subprocess, time
import numpy as np

# (opcional) si usás TensorFlow/Keras
try:
    import tensorflow as tf
except Exception:
    tf = None

def set_global_determinism(seed: int = 42, enable_tf: bool = True):
    """Fija semillas y banderas para resultados reproducibles."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["TF_DETERMINISTIC_OPS"] = "1"         # TF determinista
    os.environ["TF_CUDNN_DETERMINISTIC"] = "1"
    os.environ["OMP_NUM_THREADS"] = "1"              # evita non-determinism por paralelismo
    os.environ["OPENBLAS_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
    os.environ["NUMEXPR_NUM_THREADS"] = "1"

    random.seed(seed)
    np.random.seed(seed)

    if enable_tf and tf is not None:
        try:
            tf.random.set_seed(seed)
            # Forzar single-thread en TF para mayor determinismo
            tf.config.threading.set_intra_op_parallelism_threads(1)
            tf.config.threading.set_inter_op_parallelism_threads(1)
        except Exception as e:
            print(f"[WARN] No se pudo fijar determinismo TF: {e}")

    print(f"[OK] Semillas y entorno fijados con seed={seed}")

# --- GPU safety para TensorFlow (evita OOM y calores) ---
if tf is not None:
    try:
        gpus = tf.config.list_physical_devices('GPU')
        if gpus:
            for g in gpus:
                tf.config.experimental.set_memory_growth(g, True)
            print(f"[TF] GPU detectada ({len(gpus)}), memory_growth=True")
        else:
            print("[TF] Sin GPU; correrá en CPU.")
    except Exception as e:
        print(f"[TF] No pude configurar memory_growth: {e}")    

def lib_versions():
    """Devuelve versiones útiles para reproducibilidad."""
    vers = {"python": sys.version.split()[0], "platform": platform.platform()}
    def _get_ver(mod, name):
        try:
            import importlib
            m = importlib.import_module(mod)
            vers[name] = getattr(m, "__version__", "unknown")
        except Exception:
            vers[name] = "not_installed"
    _get_ver("numpy","numpy"); _get_ver("pandas","pandas"); _get_ver("scipy","scipy")
    _get_ver("sklearn","scikit_learn"); _get_ver("xgboost","xgboost")
    _get_ver("lightgbm","lightgbm"); _get_ver("catboost","catboost")
    _get_ver("tensorflow","tensorflow"); _get_ver("keras","keras")
    return vers

def save_run_metadata(path_json:str,
                      seed:int,
                      feature_cols:list=None,
                      target_col:str=None,
                      extra:dict=None):
    """Guarda metadatos mínimos de la corrida."""
    meta = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "seed": seed,
        "versions": lib_versions(),
        "feature_cols": list(feature_cols) if feature_cols is not None else None,
        "target_col": target_col,
    }
    if extra:
        meta["extra"] = extra
    os.makedirs(os.path.dirname(path_json) or ".", exist_ok=True)
    with open(path_json, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)
    print(f"[OK] Metadatos guardados en: {path_json}")

# ======= INVOCACIÓN TEMPRANA (llamar una sola vez, bien arriba del pipeline) =======
GLOBAL_SEED = 42
set_global_determinism(GLOBAL_SEED, enable_tf=True)


[TF] GPU detectada (1), memory_growth=True
[OK] Semillas y entorno fijados con seed=42


In [5]:
# ================================================================================
# BLOQUE 4: CARGA, LIMPIEZA INICIAL Y ENRIQUECIMIENTO CON DATOS DE SECTOR
# ================================================================================
# --- 4.1: Carga y Limpieza Inicial ---
# Cargar los datos (reemplaza con tu ruta de archivo)
file_path = 'C:/Users/Andy/OneDrive/Desktop/MCD/Tesis/Datos_fuente_Bloomberg/en valores/serie completa 2014-2024/Dataset/dataset_oct_2014-set_2024.xlsx'
df = pd.read_excel(file_path, sheet_name='dataset')

# Limpieza y Conversión de Tipos
df['Non farm payrolls_Exp_mediana'] = df['Non farm payrolls_Exp_mediana'].astype(str).str.replace('k', '').astype(float)
df['Non farm payrolls_Exp_promedio'] = df['Non farm payrolls_Exp_promedio'].astype(str).str.replace('k', '').astype(float)
df['Fecha'] = pd.to_datetime(df['Fecha'], format='%Y%m%d')
df['CPI'] /= 100
df['Fed Funds Rate'] /= 100

# Ordenar datos, es crucial para cálculos temporales
df = df.sort_values(['Empresa', 'Fecha']).reset_index(drop=True)
print("Datos cargados y limpieza inicial completada.")

# --- 4.2: Descarga y Unión de Datos de Sector desde Finnhub ---
print("\nIniciando descarga de datos de sector desde Finnhub...")

# Función de limpieza para convertir ticker de Bloomberg a formato estándar
def bloomberg_to_symbol(tk:str) -> str:
    return tk.split()[0].replace('.', '-')

symbols_to_fetch = df['Empresa'].map(bloomberg_to_symbol).unique().tolist()

# Cargar caché si existe
if os.path.exists(CACHE_FILE):
    sector_df = pd.read_pickle(CACHE_FILE)
    cached_symbols = set(sector_df['Symbol'])
else:
    sector_df = pd.DataFrame()
    cached_symbols = set()

records_to_add = []
tickers_to_process = [s for s in symbols_to_fetch if s not in cached_symbols]

if not tickers_to_process:
    print("  - Todos los datos de sector ya estaban en el caché.")
else:
    print(f"  - Se descargarán datos para {len(tickers_to_process)} nuevos tickers.")
    for i, symbol in enumerate(tickers_to_process, 1):
        print(f"    Descargando {symbol} ({i}/{len(tickers_to_process)})...")
        url = f"https://finnhub.io/api/v1/stock/profile2?symbol={symbol}&token={FINNHUB_API_KEY}"
        
        try:
            response = requests.get(url, timeout=15)
            response.raise_for_status()  # Lanza un error para códigos 4xx/5xx
            data = response.json()

            if data and data.get('finnhubIndustry'):
                records_to_add.append({
                    'Symbol':   symbol,
                    'Sector':   data.get('gicSector') or data.get('finnhubIndustry'), # Prioriza GICS
                    'Industry': data.get('finnhubIndustry')
                })
            else:
                print(f"    ⚠️  Sin datos de sector para {symbol}.")
        except requests.exceptions.RequestException as e:
            print(f"    ❌ Error de red para {symbol}: {e}")
        
        # Respetar el límite de tasa de la API
        if i % RATE_LIMIT == 0 and i < len(tickers_to_process):
            print(f"  - Límite de {RATE_LIMIT} llamadas alcanzado. Pausando por {SLEEP_SECONDS} segundos...")
            time.sleep(SLEEP_SECONDS)

    # Añadir los nuevos registros y guardar el caché actualizado
    if records_to_add:
        new_data_df = pd.DataFrame(records_to_add)
        sector_df = pd.concat([sector_df, new_data_df], ignore_index=True).drop_duplicates('Symbol')
        sector_df.to_pickle(CACHE_FILE)
        print(f"\n{len(records_to_add)} nuevos símbolos añadidos al caché -> {CACHE_FILE}")

# Unir los datos de sector al DataFrame principal
if not sector_df.empty:
    df['Symbol_key'] = df['Empresa'].map(bloomberg_to_symbol)
    df = pd.merge(
        df,
        sector_df,
        left_on='Symbol_key',
        right_on='Symbol',
        how='left'
    )
    df.drop(columns=['Symbol_key', 'Symbol'], inplace=True)
    print("\nDatos de Sector e Industria unidos al DataFrame principal.")
else:
    print("\nNo se pudieron obtener datos de sector.")

Datos cargados y limpieza inicial completada.

Iniciando descarga de datos de sector desde Finnhub...
  - Todos los datos de sector ya estaban en el caché.

Datos de Sector e Industria unidos al DataFrame principal.


In [6]:
# ===============================================================================
# BLOQUE 5 · Normalización de fechas a FIN DE MES (alineación global)
#   - Asegura que todo el pipeline (riesgo, anclaje Q-2, señales, buckets, splits)
#     trabaje en la misma grilla temporal (MonthEnd).
# ===============================================================================

print("\n🗓️  Normalizando fechas a fin de mes…")

# 1) Forzar Fecha a MonthEnd (fin de mes)
df['Fecha'] = pd.to_datetime(df['Fecha'])
df['Fecha'] = df['Fecha'].dt.to_period('M').dt.to_timestamp('M')

# 2) Resolver posibles duplicados (Empresa, Fecha) manteniendo la última fila del mes
dup_count = df.duplicated(subset=['Empresa', 'Fecha'], keep=False).sum()
if dup_count:
    df = (
        df.sort_values(['Empresa', 'Fecha'])
          .groupby(['Empresa', 'Fecha'], as_index=False)
          .tail(1)
          .reset_index(drop=True)
    )
    print(f"  · Se resolvieron {dup_count} duplicados (Empresa, Fecha).")

# 3) Orden final de seguridad
df = df.sort_values(['Empresa', 'Fecha']).reset_index(drop=True)

# 4) Chequeo rápido
assert not df.duplicated(subset=['Empresa', 'Fecha']).any(), "Quedaron duplicados (Empresa, Fecha)."

print("  · Fechas normalizadas a fin de mes y dataset ordenado.")



🗓️  Normalizando fechas a fin de mes…
  · Fechas normalizadas a fin de mes y dataset ordenado.


In [7]:
# --- 6.1: Conteo de Empresas por Sector ---
sector_counts = (
    df.drop_duplicates(['Empresa', 'Sector'])  # ➊ deja una sola fila por empresa-sector
      .groupby('Sector')['Empresa']            # ➋ agrupa por sector
      .nunique()                               # ➌ cuenta empresas únicas
      .reset_index(name='num_empresas')        # ➍ DataFrame ordenado
      .sort_values('num_empresas', ascending=False)
)

sector_counts

,Sector,num_empresas
30,Technology,18
29,Semiconductors,17
27,Retail,17
11,Electrical Equipment,12
13,Financial Services,10
15,Health Care,10
25,Professional Services,7
14,Food Products,7
32,Trading Companies & Distributors,6
23,Media,6


In [8]:
# --- 6.2: Conteo de Empresas por Industria ---
industry_counts = (
    df.drop_duplicates(['Empresa', 'Industry'])  # ➊ deja una sola fila por empresa-industria
      .groupby('Industry')['Empresa']            # ➋ agrupa por industria
      .nunique()                               # ➌ cuenta empresas únicas
      .reset_index(name='num_empresas')        # ➍ DataFrame ordenado
      .sort_values('num_empresas', ascending=False)
)

industry_counts

,Industry,num_empresas
30,Technology,18
29,Semiconductors,17
27,Retail,17
11,Electrical Equipment,12
13,Financial Services,10
15,Health Care,10
25,Professional Services,7
14,Food Products,7
32,Trading Companies & Distributors,6
23,Media,6


In [9]:
# ========================================================================
# BLOQUE 7 · Forzar Industry  →  11 Sectores GICS
# ========================================================================

# --- 1.  Diccionario completo ------------------------------------------
INDUSTRY_TO_GICS = {
    # Information Technology
    'Technology':                      'Information Technology',
    'Semiconductors':                  'Information Technology',

    # Consumer Discretionary
    'Retail':                          'Consumer Discretionary',
    'Consumer products':               'Consumer Discretionary',
    'Hotels, Restaurants & Leisure':   'Consumer Discretionary',
    'Auto Components':                 'Consumer Discretionary',
    'Textiles, Apparel & Luxury Goods':'Consumer Discretionary',
    'Leisure Products':                'Consumer Discretionary',
    'Distributors':                    'Consumer Discretionary',

    # Industrials
    'Electrical Equipment':            'Industrials',
    'Machinery':                       'Industrials',
    'Trading Companies & Distributors':'Industrials',
    'Road & Rail':                     'Industrials',
    'Commercial Services & Supplies':  'Industrials',
    'Industrial Conglomerates':        'Industrials',
    'Building':                        'Industrials',
    'Aerospace & Defense':             'Industrials',
    'Logistics & Transportation':      'Industrials',
    'Professional Services':           'Industrials',

    # Financials
    'Financial Services':              'Financials',
    'Insurance':                       'Financials',

    # Health Care
    'Health Care':                     'Health Care',
    'Biotechnology':                   'Health Care',
    'Life Sciences Tools & Services':  'Health Care',

    # Communication Services
    'Media':                           'Communication Services',
    'Communications':                  'Communication Services',

    # Consumer Staples
    'Food Products':                   'Consumer Staples',
    'Beverages':                       'Consumer Staples',

    # Utilities
    'Utilities':                       'Utilities',

    # Materials
    'Chemicals':                       'Materials',
    'Metals & Mining':                 'Materials',
    'Construction':                    'Materials',

    # Energy
    'Energy':                          'Energy',

    # Real Estate
    'Real Estate':                     'Real Estate',
}

# --- 2.  Asignar SIEMPRE a partir de Industry --------------------------
df['Sector_GICS'] = df['Industry'].map(INDUSTRY_TO_GICS).fillna('Unknown')

# --- 3.  Chequeo rápido -------------------------------------------------
print("\nDistribución después del mapeo (deberían ser ≤ 11):")
display(
    df.drop_duplicates(['Empresa','Sector_GICS'])
      .groupby('Sector_GICS')['Empresa']
      .nunique()
      .sort_values(ascending=False)
      .to_frame('empresas')
)


Distribución después del mapeo (deberían ser ≤ 11):


,empresas
Sector_GICS,
Industrials,50
Consumer Discretionary,38
Information Technology,35
Health Care,13
Consumer Staples,11
Financials,11
Communication Services,8
Materials,5
Real Estate,4


In [10]:
# ==============================================================================
# BLOQUE 8: Descarga y Procesamiento de Tasa Libre de Riesgo (FRED)
# ==============================================================================

# --- CONFIGURACIÓN ---
# Símbolo de FRED para la T-Bill a 3 meses (tasa anualizada en %)
fred_symbol = 'TB3MS'
start_date = datetime.datetime(2014, 1, 1)
end_date = datetime.datetime.now()  # Hasta la fecha actual

print(f"Descargando datos para '{fred_symbol}' desde FRED (desde {start_date.date()})...")

try:
    # 1) Descargar datos diarios/mensuales según disponibilidad
    t_bills_rate_raw = web.DataReader(fred_symbol, 'fred', start_date, end_date)

    # 2) Remuestrear a FIN DE MES y rellenar huecos hacia adelante (valor más reciente conocido)
    t_bills_rate = t_bills_rate_raw.resample('M').last().ffill()
    print("  · Fechas re-muestreadas a fin de mes y huecos rellenados (ffill).")

    # 3) Convertir a decimal y calcular retorno mensual (anual/12)
    t_bills_rate[fred_symbol] = t_bills_rate[fred_symbol] / 100.0
    t_bills_rate['risk_free_monthly_return'] = t_bills_rate[fred_symbol] / 12.0
    print("  · Tasa convertida a decimal y retorno mensual calculado (anual/12).")

    # 4) Index y DataFrame final prolijos
    t_bills_rate.index.name = 'Fecha'
    df_risk_free = t_bills_rate[['risk_free_monthly_return']].copy()

    # 5) (Opcional) Chequeo rápido de NaNs
    n_nans = int(df_risk_free['risk_free_monthly_return'].isna().sum())
    if n_nans == 0:
        print("  · No hay NaNs en 'risk_free_monthly_return'.")
    else:
        print(f"  · Advertencia: Hay {n_nans} NaNs en 'risk_free_monthly_return' tras el procesamiento.")

    print("\n--- Datos de Tasa Libre de Riesgo listos para usar ---")
    print("Primeras filas:")
    print(df_risk_free.head())
    print("\nÚltimas filas:")
    print(df_risk_free.tail())

    # --- (Opcional) Guardar en disco ---
    # df_risk_free.to_csv('risk_free_returns_monthly_2014_present.csv')
    # print("\nDatos guardados en 'risk_free_returns_monthly_2014_present.csv'")

except Exception as e:
    print(f"\nERROR al descargar o procesar los datos de FRED: {e}")
    print("Verifica tu conexión a internet o si la librería pandas-datareader está actualizada.")
    df_risk_free = None  # Indica fallo

# --- CÓMO USARLO EN TU BACKTEST ---
#
# 1) Asegúrate de que el índice de tu DataFrame de resultados (spr_df)
#    también sea DatetimeIndex y esté a fin de mes.
#
# 2) Merge + ffill (modo estándar, suficiente para excess returns del mes t):
#    spr_df = spr_df.merge(df_risk_free, left_index=True, right_index=True, how='left')
#    spr_df['risk_free_monthly_return'].ffill(inplace=True)
#    spr_df['excess_return_net'] = spr_df['spread_net'] - spr_df['risk_free_monthly_return']
#
# 3) (Opcional, ultra-conservador) usar RF con un mes de lag:
#    df_risk_free['rf_m_lag1'] = df_risk_free['risk_free_monthly_return'].shift(1)
#    spr_df = spr_df.merge(df_risk_free[['rf_m_lag1']], left_index=True, right_index=True, how='left')
#    spr_df['rf_m_lag1'].ffill(inplace=True)
#    spr_df['excess_return_net'] = spr_df['spread_net'] - spr_df['rf_m_lag1']
#
# Nota: En ambos casos no hay fuga de información hacia el ranking cross-sectional,
# porque la RF desplaza a todos por igual ese mes. Usa el modo lag si quieres timing
# ultra conservador o si vas a emplear RF como feature del modelo.


Descargando datos para 'TB3MS' desde FRED (desde 2014-01-01)...
  · Fechas re-muestreadas a fin de mes y huecos rellenados (ffill).
  · Tasa convertida a decimal y retorno mensual calculado (anual/12).
  · No hay NaNs en 'risk_free_monthly_return'.

--- Datos de Tasa Libre de Riesgo listos para usar ---
Primeras filas:
            risk_free_monthly_return
Fecha                               
2014-01-31                  0.000033
2014-02-28                  0.000042
2014-03-31                  0.000042
2014-04-30                  0.000025
2014-05-31                  0.000025

Últimas filas:
            risk_free_monthly_return
Fecha                               
2025-03-31                  0.003500
2025-04-30                  0.003508
2025-05-31                  0.003542
2025-06-30                  0.003525
2025-07-31                  0.003542


In [11]:
# ==============================================================================
# BLOQUE 9 · FEATURE ENGINEERING AVANZADO
#   – Momentum (1-3-6-12 m)
#   – Volatilidad (3-6-12 m)
#   – Sorpresas macro
#   – Momentum 18 m + Vol-de-Vol 6 m
#   – PE-zscore 6 m
#   – Momentum sectorial (seguro, usando GICS/fallback)
#   – Dummies sector GICS  / Dummy pandemia
# ==============================================================================

print("\nIniciando Feature Engineering Avanzado…")

# ────────────────────────────────────────────────────────────────────────────────
# 0)  Asegurar listas y parámetros base
# ────────────────────────────────────────────────────────────────────────────────
if 'features_mercado' not in locals() or not isinstance(features_mercado, list):
    features_mercado = []

if 'MIN_PERIODS_WINDOW' not in locals():
    MIN_PERIODS_WINDOW = 3

if 'VALUATION_MOMENTUM_WINDOW' not in locals():
    VALUATION_MOMENTUM_WINDOW = 6

# ────────────────────────────────────────────────────────────────────────────────
# 9.1  Momentum y volatilidad de precios + sorpresas macro
# ────────────────────────────────────────────────────────────────────────────────

# --- Momentum de precios -------------------------------------------------------
print("  · Calculando momentum de precios...")
for p in (1, 3, 6, 12):
    col = f'ret_P_Share_{p}m_base'
    df[col] = df.groupby('Empresa')['P_Share'].pct_change(p)
    features_mercado.append(col)

# --- Volatilidad rolling -------------------------------------------------------
print("  · Calculando volatilidad de precios...")
for w in (3, 6, 12):
    col = f'vol_P_Share_{w}m_base'
    # rolling std por empresa manteniendo alineación
    df[col] = (
        df.groupby('Empresa', group_keys=False)['P_Share']
          .apply(lambda s: s.rolling(window=w, min_periods=MIN_PERIODS_WINDOW).std())
    )
    features_mercado.append(col)

# --- Sorpresas macro -----------------------------------------------------------
print("  · Calculando sorpresas macro...")
df['dif_CPI_mediana'] =  (df['CPI'] - df['CPI_Exp_mediana']) * 10_000
df['dif_FFR_mediana'] = ((df['Fed Funds Rate'] - df['Fed Funds Rate_Exp_mediana']) * 10_000)
df['dif_NFP_mediana'] =   df['Non farm payrolls'] - df['Non farm payrolls_Exp_mediana']

# Imputación conservadora para features base recién creadas
base_cols = [c for c in df.columns if c.endswith('_base') or c.startswith('dif_')]
df[base_cols] = df[base_cols].fillna(0.0)
print("  · Features base creadas e imputadas (NaN→0).")

# ────────────────────────────────────────────────────────────────────────────────
# 9-plus  · Momentum 18 m (ret_18m_lag1)  +  Vol-de-Vol 6 m (vov_6m_lag1)
# ────────────────────────────────────────────────────────────────────────────────
print("  · Añadiendo momentum 18 m y vol-de-vol 6 m…")

# Momentum 18m con un lag adicional (info conocida al cierre t-1)
df['ret_18m_tmp']  = df.groupby('Empresa')['P_Share'].pct_change(18)
df['ret_18m_lag1'] = df.groupby('Empresa')['ret_18m_tmp'].shift(1).fillna(0.0)
df.drop(columns='ret_18m_tmp', inplace=True)
features_mercado.append('ret_18m_lag1')

# Vol-de-Vol 6m (std de la std rolling 6m), winsorizado 1-99%, luego lag(1)
vol6 = (
    df.groupby('Empresa', group_keys=False)['P_Share']
      .apply(lambda s: s.rolling(window=6, min_periods=3).std())
)
vov6 = (
    vol6.groupby(df['Empresa'], group_keys=False)
        .apply(lambda s: s.rolling(window=6, min_periods=3).std())
)

# winsorizar por percentiles globales (robusto)
p1, p99 = vov6.quantile([.01, .99])
vov6_clip = vov6.clip(lower=p1 if pd.notna(p1) else None,
                      upper=p99 if pd.notna(p99) else None)

df['vov_6m_lag1'] = vov6_clip.groupby(df['Empresa']).shift(1).fillna(0.0)
features_mercado.append('vov_6m_lag1')
print("    · ret_18m_lag1 y vov_6m_lag1 añadidas.")

# ────────────────────────────────────────────────────────────────────────────────
# 9.2  Z-score de valoración (P/E winsorizado, ventana 6 m)
# ────────────────────────────────────────────────────────────────────────────────
print("  · Calculando Z-score de valoración (PE_zscore_6m)...")

from scipy.stats.mstats import winsorize

def winsor_1pct(series: pd.Series) -> pd.Series:
    """Winsoriza la serie al 1 % por cola, conservando índice."""
    if series.dropna().shape[0] < 2:
        return series
    arr = winsorize(series.dropna().astype(float), limits=[.01, .01])
    return pd.Series(arr, index=series.dropna().index, dtype='float64').reindex(series.index)

# 1) P/E winsorizado empresa-a-empresa
df['P_E_wins'] = (
    df.groupby('Empresa', group_keys=False)['P_E']
      .apply(winsor_1pct)
)

# 2) Mediana y MAD móvil (6m) por empresa
eps = 1e-6
def rolling_mad(x):
    med = np.median(x)
    return np.median(np.abs(x - med)) + eps

df['median_6m'] = (
    df.groupby('Empresa', group_keys=False)['P_E_wins']
      .apply(lambda s: s.rolling(window=VALUATION_MOMENTUM_WINDOW,
                                 min_periods=MIN_PERIODS_WINDOW).median())
)
df['mad_6m'] = (
    df.groupby('Empresa', group_keys=False)['P_E_wins']
      .apply(lambda s: s.rolling(window=VALUATION_MOMENTUM_WINDOW,
                                 min_periods=MIN_PERIODS_WINDOW).apply(rolling_mad, raw=True))
)

# 3) Z-score robusto
df['PE_zscore_6m'] = (df['P_E_wins'] - df['median_6m']) / (df['mad_6m'] * 1.4826)
df['PE_zscore_6m'].replace([np.inf, -np.inf], 0.0, inplace=True)
df['PE_zscore_6m'].fillna(0.0, inplace=True)

# 4) Dummy historial corto
df['is_short_hist_zscore'] = (df.groupby('Empresa').cumcount() < VALUATION_MOMENTUM_WINDOW).astype(int)

# Limpieza temporales
df.drop(columns=['P_E_wins', 'median_6m', 'mad_6m'], inplace=True)
features_mercado += ['PE_zscore_6m', 'is_short_hist_zscore']
print("  · Z-score de valoración (PE_zscore_6m) calculado.")

# ────────────────────────────────────────────────────────────────────────────────
# 9.2-bis  Momentum sectorial (seguro) — usa Sector_GICS con fallback
# ────────────────────────────────────────────────────────────────────────────────
print("  · Calculando señal de momentum sectorial (versión segura, GICS/fallback)…")

# Columna de sector de referencia: GICS si existe; si no, Sector; si no, 'Unknown'
df['Sector_ref'] = (
    (df['Sector_GICS'] if 'Sector_GICS' in df.columns else pd.Series(index=df.index))
        .fillna(df['Sector'] if 'Sector' in df.columns else None)
        .fillna('Unknown')
)

# Usar precios lag-1 (info disponible al cierre t-1)
df['P_Share_l1'] = df.groupby('Empresa')['P_Share'].shift(1)

tmp = (
    df[['Fecha', 'Empresa', 'Sector_ref', 'P_Share_l1']]
      .dropna(subset=['Sector_ref', 'P_Share_l1'])
      .sort_values(['Empresa', 'Fecha'])
)

# Retornos sobre P_Share_l1 (→ info hasta t-1)
tmp['ret1']  = tmp.groupby('Empresa')['P_Share_l1'].pct_change(1)
tmp['ret12'] = tmp.groupby('Empresa')['P_Share_l1'].pct_change(12)

# Promedio por sector y score
sector_mom = (
    tmp.groupby(['Fecha', 'Sector_ref'])[['ret12', 'ret1']].mean().reset_index()
)
sector_mom['sec_score'] = 0.5 * (sector_mom['ret12'] - sector_mom['ret1'])

# Z-score cross-sectional por fecha
def _z(s):
    mu, sd = s.mean(), s.std(ddof=0)
    return (s - mu) / (sd + 1e-9) if sd > 0 else 0.0

sector_mom['sec_score_z'] = sector_mom.groupby('Fecha')['sec_score'].transform(_z)

# Unir al DF principal y dejar la columna YA laggeada
df = df.merge(sector_mom[['Fecha', 'Sector_ref', 'sec_score_z']],
              on=['Fecha', 'Sector_ref'], how='left')

df['sec_score_z_lag1'] = df['sec_score_z'].fillna(0.0)
df.drop(columns=['sec_score_z'], inplace=True)
features_mercado.append('sec_score_z_lag1')
print("  · Señal sectorial segura incorporada: sec_score_z_lag1")

# ────────────────────────────────────────────────────────────────────────────────
# 9.3  Dummies de sector (GICS)
# ────────────────────────────────────────────────────────────────────────────────
print("  · Creando dummies sector GICS…")
d_sec = pd.get_dummies(df['Sector_GICS'], prefix='sector', drop_first=True)
df = pd.concat([df, d_sec], axis=1)

# añadir las dummies a la lista de mercado
features_mercado += [c for c in d_sec.columns]
print(f"  · Dummies sector GICS creadas: {len(d_sec.columns)} columnas")

# ────────────────────────────────────────────────────────────────────────────────
# 9.4  Dummy pandemia (mar-20 → jun-21)
# ────────────────────────────────────────────────────────────────────────────────
print("  · Creando dummy de pandemia...")
df['pandemic_dummy'] = ((df['Fecha'] >= '2020-03-01') & (df['Fecha'] <= '2021-06-30')).astype(int)
print("  · Dummy pandemia creada.")

# ────────────────────────────────────────────────────────────────────────────────
# 9.5  Consolidar lista de features de mercado (sin duplicados)
# ────────────────────────────────────────────────────────────────────────────────
features_mercado = list(dict.fromkeys(features_mercado))  # dedup preservando orden

print("\n— Fin Bloque 4 · DataFrame listo para los lags contables —")
try:
    print("Ejemplo de filas:")
    print(df.head(5))
except Exception:
    pass



Iniciando Feature Engineering Avanzado…
  · Calculando momentum de precios...
  · Calculando volatilidad de precios...
  · Calculando sorpresas macro...
  · Features base creadas e imputadas (NaN→0).
  · Añadiendo momentum 18 m y vol-de-vol 6 m…
    · ret_18m_lag1 y vov_6m_lag1 añadidas.
  · Calculando Z-score de valoración (PE_zscore_6m)...
  · Z-score de valoración (PE_zscore_6m) calculado.
  · Calculando señal de momentum sectorial (versión segura, GICS/fallback)…
  · Señal sectorial segura incorporada: sec_score_z_lag1
  · Creando dummies sector GICS…
  · Dummies sector GICS creadas: 10 columnas
  · Creando dummy de pandemia...
  · Dummy pandemia creada.

— Fin Bloque 4 · DataFrame listo para los lags contables —
Ejemplo de filas:
          Empresa      Fecha      P_E     P_B     P_S  P_Share       ROCE  \
0  AAPL US Equity 2014-10-31  16.8500  5.6796  3.6047   27.000  32.246867   
1  AAPL US Equity 2014-11-30  18.5553  6.2544  3.9695   29.733  32.929333   
2  AAPL US Equity 2014-

In [12]:
df.head(20)

,Empresa,Fecha,P_E,P_B,P_S,P_Share,ROCE,EBIT,Total Activos,Deuda a LP,...,sector_Consumer Staples,sector_Energy,sector_Financials,sector_Health Care,sector_Industrials,sector_Information Technology,sector_Materials,sector_Real Estate,sector_Utilities,pandemic_dummy
0,AAPL US Equity,2014-10-31,16.8500,5.6796,3.6047,27.000,32.246867,10576.333333,225626.333333,29015.666667,...,False,False,False,False,False,True,False,False,False,0
1,AAPL US Equity,2014-11-30,18.5553,6.2544,3.9695,29.733,32.929333,10870.666667,228732.666667,29001.333333,...,False,False,False,False,False,True,False,False,False,0
2,AAPL US Equity,2014-12-31,14.9215,5.2147,3.2904,27.595,33.611800,11165.000000,231839.000000,28987.000000,...,False,False,False,False,False,True,False,False,False,0
3,AAPL US Equity,2015-01-31,15.8381,5.5350,3.4925,29.290,34.123267,15525.333333,241857.333333,30159.333333,...,False,False,False,False,False,True,False,False,False,0
4,AAPL US Equity,2015-02-28,17.3656,6.0689,3.8294,32.115,34.634733,19885.666667,251875.666667,31331.666667,...,False,False,False,False,False,True,False,False,False,0
5,AAPL US Equity,2015-03-31,15.3999,5.5579,3.4464,31.108,35.146200,24246.000000,261894.000000,32504.000000,...,False,False,False,False,False,True,False,False,False,0
6,AAPL US Equity,2015-04-30,15.4890,5.5900,3.4664,31.288,36.221300,22256.666667,261660.666667,35026.666667,...,False,False,False,False,False,True,False,False,False,0
7,AAPL US Equity,2015-05-31,16.1239,5.8192,3.6085,32.570,37.296400,20267.333333,261427.333333,37549.333333,...,False,False,False,False,False,True,False,False,False,0
8,AAPL US Equity,2015-06-30,14.4905,5.6940,3.2548,31.356,38.371500,18278.000000,261194.000000,40072.000000,...,False,False,False,False,False,True,False,False,False,0
9,AAPL US Equity,2015-07-31,14.0140,5.5067,3.1477,30.325,39.296467,16879.666667,265179.666667,42521.000000,...,False,False,False,False,False,True,False,False,False,0


In [13]:
# ===============================================================================
# BLOQUE 10 · ANCLAJE CONTABLE (Q-2) + TARGET + LISTAS DE FEATURES (SIMPLIFICADO)
# ===============================================================================

print("\n⏩  BLOQUE 5  –  Anclaje contable y construcción del dataset final…")

# ───────────────────────────────────────────────────────────────────────────────
# 10.0 · DEFINICIÓN DE LAS LISTAS DE FEATURES
# ───────────────────────────────────────────────────────────────────────────────

# 1) CONTABLES (nombres limpios desde el Excel)
features_contables = [
    'ROCE', 'ROA', 'EBIT', 'Total Activos', 'Deuda a LP',
    'Beneficio neto', 'ROI', 'EV', 'Cap de mercado',
    'Deuda a CP', 'Efectivo y equiv'
]
# Usar solo las columnas que realmente existen
features_contables = [col for col in features_contables if col in df.columns]

# 2) MERCADO / MOMENTUM (consolidar sin duplicar, preservando orden)
base_market = [
    'P_E','P_B','P_S',
    'ret_P_Share_1m_base','ret_P_Share_3m_base','ret_P_Share_6m_base','ret_P_Share_12m_base',
    'vol_P_Share_3m_base','vol_P_Share_6m_base','vol_P_Share_12m_base',
    'PE_zscore_6m','is_short_hist_zscore',
    'ret_18m_lag1','vov_6m_lag1','sec_score_z_lag1'
]

if 'features_mercado' in locals() and isinstance(features_mercado, list):
    # dedup preservando el orden
    seen = set()
    features_mercado = [x for x in (features_mercado + base_market) if not (x in seen or seen.add(x))]
else:
    features_mercado = base_market[:]

# Añadir dummies sectoriales creadas en Bloque 4.3 (si existen)
features_mercado += [c for c in df.columns if c.startswith('sector_')]

# 3) MACRO
features_macro = []

print(f"    · {len(features_contables):2d} contables, "
      f"{len(features_mercado):2d} mercado, "
      f"{len(features_macro):2d} macro definidos.")

# ───────────────────────────────────────────────────────────────────────────────
# 10.1 · ANCLAJE CONTABLE (Q-2)
# ───────────────────────────────────────────────────────────────────────────────
# Mapea cada mes (t) con el último dato contable disponible de t-2 trimestres
df['periodo_trimestre'] = df['Fecha'].dt.to_period('Q')
QUARTERLY_OFFSET = 2
df['periodo_informe'] = df['periodo_trimestre'] - QUARTERLY_OFFSET

# Tomar el último registro de cada trimestre por empresa (fin de trimestre)
df_cont_q = (
    df.sort_values('Fecha')
      .groupby(['Empresa', 'periodo_trimestre'])
      .tail(1)
      .copy()
)

# Eliminar columnas contables actuales antes del merge (evita sombras)
df.drop(columns=features_contables, errors='ignore', inplace=True)

# Merge: trae las contables del trimestre (t-2) a cada fila mensual en t
df = pd.merge(
    df,
    df_cont_q[['Empresa', 'periodo_trimestre'] + features_contables],
    left_on=['Empresa', 'periodo_informe'],
    right_on=['Empresa', 'periodo_trimestre'],
    how='left'
)

# Limpieza de columnas auxiliares
df.drop(columns=[c for c in df.columns if c.startswith('periodo_')], inplace=True, errors='ignore')
print("    · Contables anclados correctamente (Q-2).")

# ───────────────────────────────────────────────────────────────────────────────
# 10.2 · VARIABLE OBJETIVO (retorno futuro a 1 mes)
# ───────────────────────────────────────────────────────────────────────────────
TARGET_HORIZON = 1
TARGET_COL = f"retorno_futuro_{TARGET_HORIZON}m"

df[TARGET_COL] = df.groupby('Empresa')['P_Share'].shift(-TARGET_HORIZON) / df['P_Share'] - 1

rows_before = len(df)
df.dropna(subset=[TARGET_COL], inplace=True)
print(f"    · Target creado. Filas eliminadas por NaN en target: {rows_before - len(df):,}")

print("\n✅  Fin Bloque 5 – DataFrame listo para el Bloque 6 (lags finales).")

# ───────────────────────────────────────────────────────────────────────────────
# 10.3 · GUARDADO DE METADATOS DE LA CORRIDA (reproducibilidad)
# ───────────────────────────────────────────────────────────────────────────────
# Si ya existe predictor_cols_final (p.ej., tras Bloque 6), lo guardamos.
# Si no, guardamos un snapshot de features crudas (sin lag) para trazabilidad.
if 'predictor_cols_final' in locals():
    feature_cols_snapshot = predictor_cols_final
else:
    feature_cols_snapshot = list(dict.fromkeys(features_mercado + features_macro + features_contables))

# Asegurar carpeta de artefactos
os.makedirs("artifacts", exist_ok=True)

# Guardar registro de columnas/target y versiones
save_run_metadata(
    path_json="artifacts/run_metadata.json",
    seed=GLOBAL_SEED if 'GLOBAL_SEED' in globals() else 42,
    feature_cols=feature_cols_snapshot,
    target_col=TARGET_COL,
    extra={
        "cv": {"outer": "TimeSeriesSplit", "n_splits": 7, "gap": 1},
        "scorer": "Spearman_IC",
        "notes": "Dataset ordenado cronológicamente; contables anclados Q-2; target 1m."
    }
)



⏩  BLOQUE 5  –  Anclaje contable y construcción del dataset final…
    · 11 contables, 35 mercado,  0 macro definidos.
    · Contables anclados correctamente (Q-2).
    · Target creado. Filas eliminadas por NaN en target: 180

✅  Fin Bloque 5 – DataFrame listo para el Bloque 6 (lags finales).
[OK] Metadatos guardados en: artifacts/run_metadata.json


In [14]:
features_contables

['ROCE',
 'ROA',
 'EBIT',
 'Total Activos',
 'Deuda a LP',
 'Beneficio neto',
 'ROI',
 'EV',
 'Cap de mercado',
 'Deuda a CP',
 'Efectivo y equiv']

In [15]:
df.head(20)

,Empresa,Fecha,P_E,P_B,P_S,P_Share,CPI,CPI_Exp_mediana,CPI_Exp_promedio,Fed Funds Rate,...,EBIT,Total Activos,Deuda a LP,Beneficio neto,ROI,EV,Cap de mercado,Deuda a CP,Efectivo y equiv,retorno_futuro_1m
0,AAPL US Equity,2014-10-31,16.8500,5.6796,3.6047,27.000,0.017,1.600000e-02,0.0162,0.0025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.101222
1,AAPL US Equity,2014-11-30,18.5553,6.2544,3.9695,29.733,0.013,1.600000e-02,0.0157,0.0025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.071907
2,AAPL US Equity,2014-12-31,14.9215,5.2147,3.2904,27.595,0.008,1.400000e-02,0.0142,0.0025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.061424
3,AAPL US Equity,2015-01-31,15.8381,5.5350,3.4925,29.290,-0.001,7.000000e-03,0.0069,0.0025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.096449
4,AAPL US Equity,2015-02-28,17.3656,6.0689,3.8294,32.115,0.000,-1.000000e-03,-0.0013,0.0025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.031356
5,AAPL US Equity,2015-03-31,15.3999,5.5579,3.4464,31.108,-0.001,-1.000000e-03,-0.0008,0.0025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.005786
6,AAPL US Equity,2015-04-30,15.4890,5.5900,3.4664,31.288,-0.002,1.000000e-10,0.0002,0.0025,...,11165.0,231839.0,28987.0,8467.0,29.6832,471071.7208,603277.6191,6308.0,155239.0,0.040974
7,AAPL US Equity,2015-05-31,16.1239,5.8192,3.6085,32.570,0.000,-2.000000e-03,-0.0016,0.0025,...,11165.0,231839.0,28987.0,8467.0,29.6832,471071.7208,603277.6191,6308.0,155239.0,-0.037274
8,AAPL US Equity,2015-06-30,14.4905,5.6940,3.2548,31.356,0.001,1.000000e-03,0.0006,0.0025,...,11165.0,231839.0,28987.0,8467.0,29.6832,471071.7208,603277.6191,6308.0,155239.0,-0.032880
9,AAPL US Equity,2015-07-31,14.0140,5.5067,3.1477,30.325,0.002,1.000000e-03,0.0015,0.0025,...,24246.0,261894.0,32504.0,18024.0,28.3788,522601.5018,668533.0938,3899.0,177955.0,-0.070404


In [16]:
# ===============================================================================
# BLOQUE DIAGNÓSTICO · Validaciones de anclaje y target
# ===============================================================================

print("\n🕵️  Iniciando validaciones…")

# --- ASUNCIONES ---
# df: DataFrame que resulta del Bloque 5 (ya tiene los contables '_l' anclados).
# df_cont_q: DataFrame creado en el Bloque 5 que contiene los datos originales.
# features_contables: Lista de features contables (solo con sufijo '_l').
# TARGET_COL: Nombre de la columna objetivo.
# TARGET_HORIZON: Horizonte de la variable objetivo.

# --------------------------------------------------------------------------- #
# 1) ¿Cada fila quedó ligada al trimestre correcto?
# --------------------------------------------------------------------------- #
def esperado(fecha):
    """Trimestre que *debería* usarse (= fecha.to_period('Q') - 2)."""
    return fecha.to_period('Q') - 2

df['_trimestre_esperado'] = df['Fecha'].apply(esperado)
df['_trimestre_utilizado'] = df['Fecha'].dt.to_period('Q') - 2

mismatch = df.loc[df['_trimestre_esperado'] != df['_trimestre_utilizado'],
                  ['Empresa', 'Fecha', '_trimestre_esperado', '_trimestre_utilizado']]

if mismatch.empty:
    print("✅ Chequeo 1 – Cada fila emplea el trimestre contable correcto.")
else:
    print("⚠️ Chequeo 1 – Hay filas con trimestre mal anclado:")
    display(mismatch.head())

df.drop(columns=['_trimestre_esperado', '_trimestre_utilizado'], inplace=True)

# --------------------------------------------------------------------------- #
# 2) Vista rápida de 3 empresas al azar
# --------------------------------------------------------------------------- #
np.random.seed(42)
sel_empresas = np.random.choice(df['Empresa'].unique(), size=3, replace=False)

# --- CORRECCIÓN: Usar solo las columnas '_l' que ahora existen en df ---
cont_cols_demo = ['ROA', 'Total Activos']
# Asegurarse de que las columnas de demostración realmente existan en el DataFrame
cont_cols_demo_exist = [col for col in cont_cols_demo if col in df.columns]
cols_demo = ['Empresa', 'Fecha', 'P_Share'] + cont_cols_demo_exist

muestra = (df.loc[df['Empresa'].isin(sel_empresas), cols_demo]
             .sort_values(['Empresa', 'Fecha'])
             .groupby('Empresa', group_keys=False)
             .head(15)
             .reset_index(drop=True))

print("\n🔍 Chequeo 2 – 3 empresas al azar (primeras 15 fechas c/u):")
display(muestra)

# --------------------------------------------------------------------------- #
# 3) ¿Los contables cambian solo en el mes en que llega el nuevo informe?
# --------------------------------------------------------------------------- #
violaciones = []
# El bucle ahora itera sobre 'features_contables', que solo contiene las '_l'
for col in features_contables:
    if col in df.columns:
        cambia = df.groupby('Empresa')[col].diff().abs() > 1e-12
        bad = (cambia.groupby([df['Empresa'], df['Fecha'].dt.to_period('Q')])
                     .sum()
                     .reset_index(name='n_cambios')
                     .query('n_cambios > 1'))
        if not bad.empty:
            bad['col'] = col
            violaciones.append(bad)

if not violaciones:
    print("✅ Chequeo 3 – Sin look-ahead: los contables solo cambian cuando toca.")
else:
    print("⚠️ Chequeo 3 – Algunos contables cambian antes de tiempo:")
    display(pd.concat(violaciones).head())

# --------------------------------------------------------------------------- #
# 4) Consistencia del target
# --------------------------------------------------------------------------- #
chk = df[['Empresa', 'Fecha', 'P_Share', TARGET_COL]].copy()
chk['P_Share_futuro'] = chk.groupby('Empresa')['P_Share'].shift(-TARGET_HORIZON)
chk['reconstruido'] = (1 + chk[TARGET_COL]) * chk['P_Share']

err = (chk['reconstruido'] - chk['P_Share_futuro']).abs()
if err.dropna().max() < 1e-6:
    print(f"✅ Chequeo 4 – El target ({TARGET_HORIZON}m) se reconstruye sin error numérico.")
else:
    print("⚠️ Chequeo 4 – Discrepancias en la reconstrucción del target:")
    display(chk.loc[err > 1e-6, ['Empresa', 'Fecha', 'P_Share',
                                 'P_Share_futuro', TARGET_COL]].head())

print("\n🟢 Validaciones finalizadas.\n")


🕵️  Iniciando validaciones…
✅ Chequeo 1 – Cada fila emplea el trimestre contable correcto.

🔍 Chequeo 2 – 3 empresas al azar (primeras 15 fechas c/u):


,Empresa,Fecha,P_Share,ROA,Total Activos
0,CENT US Equity,2014-10-31,6.2457,NaN,NaN
1,CENT US Equity,2014-11-30,5.7291,NaN,NaN
2,CENT US Equity,2014-12-31,6.8718,NaN,NaN
3,CENT US Equity,2015-01-31,6.6135,NaN,NaN
4,CENT US Equity,2015-02-28,7.0910,NaN,NaN
5,CENT US Equity,2015-03-31,7.7249,NaN,NaN
6,CENT US Equity,2015-04-30,7.2436,0.7623,1148.727
7,CENT US Equity,2015-05-31,7.4353,0.7623,1148.727
8,CENT US Equity,2015-06-30,8.2650,0.7623,1148.727
9,CENT US Equity,2015-07-31,7.1927,1.3489,1188.963


✅ Chequeo 3 – Sin look-ahead: los contables solo cambian cuando toca.
✅ Chequeo 4 – El target (1m) se reconstruye sin error numérico.

🟢 Validaciones finalizadas.



In [17]:
# ===============================================================================
# Tabla de Diagnóstico de Anclaje
# ===============================================================================
print("\n🕵️  Generando Tabla de Diagnóstico de Anclaje (Versión Final)...")

# 1. Elige una empresa y las features a verificar
empresa_a_verificar = sel_empresas[0]
features_a_verificar = ['ROA', 'EBIT']
features_a_verificar_exist = [col for col in features_a_verificar if col in df.columns]

print(f"\nGenerando tabla para Empresa: {empresa_a_verificar}")

# 2. Seleccionar datos de la empresa del DataFrame final (con datos anclados)
df_empresa_anclado = df[df['Empresa'] == empresa_a_verificar].copy()

# 3. Seleccionar datos originales de fin de trimestre para la empresa
df_empresa_original_q = df_cont_q[df_cont_q['Empresa'] == empresa_a_verificar].copy()

# 4. Crear la tabla final
tabla_final = df_empresa_anclado[['Fecha'] + features_a_verificar_exist].copy()
tabla_final.rename(columns={col: f'{col}_Anclado' for col in features_a_verificar_exist}, inplace=True)

# 5. Traer el dato original del mes
df_original_crudo_renamed = df_empresa_original_q.rename(columns={
    col: f'{col}_Original_Mes' for col in features_a_verificar_exist
})

tabla_final = pd.merge(
    tabla_final,
    df_original_crudo_renamed[['Fecha'] + [f'{col}_Original_Mes' for col in features_a_verificar_exist]],
    on='Fecha',
    how='left'
)

# 6. Seleccionar y ordenar columnas para la visualización
cols_display = ['Fecha']
for feature in features_a_verificar_exist:
    cols_display.extend([f'{feature}_Original_Mes', f'{feature}_Anclado'])

tabla_final_display = tabla_final[cols_display].sort_values('Fecha')

# 7. Mostrar la tabla
print("\nTabla de Diagnóstico (Original Mes [Crudo] vs. Anclado [Q-2]):")
display(tabla_final_display.head(15))

print("\n🟢 Tabla de diagnóstico generada.")


🕵️  Generando Tabla de Diagnóstico de Anclaje (Versión Final)...

Generando tabla para Empresa: CENT US Equity

Tabla de Diagnóstico (Original Mes [Crudo] vs. Anclado [Q-2]):


,Fecha,ROA_Original_Mes,ROA_Anclado,EBIT_Original_Mes,EBIT_Anclado
0,2014-10-31,NaN,NaN,NaN,NaN
1,2014-11-30,NaN,NaN,NaN,NaN
2,2014-12-31,0.7623,NaN,1.373,NaN
3,2015-01-31,NaN,NaN,NaN,NaN
4,2015-02-28,NaN,NaN,NaN,NaN
5,2015-03-31,1.3489,NaN,1.138,NaN
6,2015-04-30,NaN,0.7623,NaN,1.373
7,2015-05-31,NaN,0.7623,NaN,1.373
8,2015-06-30,1.4166,0.7623,49.971,1.373
9,2015-07-31,NaN,1.3489,NaN,1.138



🟢 Tabla de diagnóstico generada.


In [18]:
# ===============================================================================
# BLOQUE 11  –  Dataset final (lags solo mercado & macro)
# ===============================================================================

print("\n⏩  BLOQUE 6  –  Generando dataset definitivo para el modelo…")

# -------------------------------------------------------------------------------
# 11.1 ·  Compilar listas finales
#  - Mercado + Macro → llevarán lag(1)
#  - Contables → se usan tal cual (ya anclados Q-2)
# -------------------------------------------------------------------------------
need_lag1 = sorted(list({*(features_mercado or []), *(features_macro or [])}))
use_as_is = sorted(features_contables or [])

print(f"    · Mercado+Macro (lag-1): {len(need_lag1)}  ·  Contables directos: {len(use_as_is)}")

# -------------------------------------------------------------------------------
# 11.2 ·  Crear/poblar columnas lag-1 para mercado+macro (evita doblar lag)
# -------------------------------------------------------------------------------
for col in need_lag1:
    if col in df.columns and not col.endswith('_lag1'):
        df[f"{col}_lag1"] = df.groupby('Empresa')[col].shift(1)

# Predictores definitivos (mercado/macro con _lag1 + contables tal cual)
predictor_cols_final = [c if c.endswith('_lag1') else f"{c}_lag1" for c in need_lag1] + use_as_is
# Deduplicar manteniendo el orden
predictor_cols_final = list(dict.fromkeys(predictor_cols_final))

# -------------------------------------------------------------------------------
# 11.3 ·  Construir df_model y limpiar NaNs
# -------------------------------------------------------------------------------
model_cols = ['Empresa', 'Fecha', TARGET_COL] + predictor_cols_final
df_model   = df[model_cols].copy()

before = len(df_model)
df_model.dropna(subset=predictor_cols_final, inplace=True)
dropped = before - len(df_model)

print(f"    · Filas eliminadas por NaNs en predictores: {dropped:,}")
print(f"    · Dataset final: {len(df_model):,} filas  ·  {len(predictor_cols_final)} features")

# -------------------------------------------------------------------------------
# 11.4 ·  Tipos numéricos consistentes (float32) para X
# -------------------------------------------------------------------------------
for col in predictor_cols_final:
    if col in df_model.columns:
        df_model[col] = pd.to_numeric(df_model[col], errors='coerce').fillna(0.0).astype('float32')

X = df_model[predictor_cols_final]
y = df_model[TARGET_COL]
groups = df_model['Empresa']

print(f"    · X listo: {X.shape[0]:,} filas × {X.shape[1]} cols | empresas={groups.nunique()}")



⏩  BLOQUE 6  –  Generando dataset definitivo para el modelo…
    · Mercado+Macro (lag-1): 25  ·  Contables directos: 11
    · Filas eliminadas por NaNs en predictores: 1,080
    · Dataset final: 20,340 filas  ·  36 features
    · X listo: 20,340 filas × 36 cols | empresas=180


In [19]:
# ======================================================================
# BLOQUE: Auxiliares para Pipelines (Ensure2D y NanFix)
# ======================================================================
from sklearn.preprocessing import FunctionTransformer
import numpy as np
import pandas as pd

def _ensure_2d(A):
    if isinstance(A, pd.DataFrame):
        A = A.to_numpy()
    A = np.asarray(A, dtype=np.float32)
    if A.ndim == 1:
        A = A.reshape(-1, 1)
    return A

ensure2d = FunctionTransformer(_ensure_2d, validate=False)

def _nan_fix(a):
    return np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0)

nan_fix = FunctionTransformer(_nan_fix, validate=False)



In [20]:
# ===============================================================================
# BLOQUE 11.x · Filtro de universo mínimo por mes (robustez cross-sectional)
#   - Evita meses “flacos” que rompen deciles/quintiles e inflan métricas.
#   - Reconstruye X, y, groups y fechas tras el filtro.
#   - Define df_model_sorted / df_sorted alineados 1:1 con X/y.
# ===============================================================================

print("\n🧹  Filtrando meses con universo insuficiente…")

UNIVERSE_MIN    = 30   # mínimo de acciones por mes para mantener el mes
MIN_PER_SECTOR  = 0    # pon >0 (p.ej. 5) si quieres exigir mínimo por sector (opcional)
USE_SECTOR_GICS = 'Sector_GICS' in df.columns

# 1) Conteo por mes del universo tras el dropna de predictores
n_by_month = (
    df_model.groupby('Fecha')['Empresa']
            .nunique()
            .rename('n_empresas')
)

# 2) Si condicionás por sector, construir conteo por mes y sector
if MIN_PER_SECTOR > 0 and USE_SECTOR_GICS:
    # Mapear Sector_GICS al df_model (índices alineados con df)
    sec_series = df.loc[df_model.index, 'Sector_GICS']
    tmp = (
        pd.concat([df_model[['Fecha','Empresa']], sec_series.rename('Sector_GICS')], axis=1)
          .dropna(subset=['Sector_GICS'])
    )
    n_by_m_sec = (
        tmp.groupby(['Fecha','Sector_GICS'])['Empresa']
           .nunique()
           .rename('n_emp_sec')
           .reset_index()
    )
    ok_by_m_sec = (
        n_by_m_sec.groupby('Fecha')['n_emp_sec']
                  .apply(lambda s: (s >= MIN_PER_SECTOR).all())
    )
    mask_month_sector_ok = ok_by_m_sec.reindex(n_by_month.index).fillna(False)
else:
    mask_month_sector_ok = pd.Series(True, index=n_by_month.index)

# 3) Meses que cumplen los mínimos
mask_month_size_ok = (n_by_month >= UNIVERSE_MIN)
good_months = n_by_month.index[mask_month_size_ok & mask_month_sector_ok]

# 4) Aplicar filtro al df_model, ordenar y resetear
df_model = df_model[df_model['Fecha'].isin(good_months)].copy()
df_model = df_model.sort_values(['Fecha','Empresa']).reset_index(drop=True)

# 4.bis) Compatibilidad → construir df_model_sorted y df_sorted alineados a df_model
df_model_sorted = df_model.copy()
df_sorted = (
    df.set_index(['Empresa','Fecha'])
      .loc[pd.MultiIndex.from_frame(df_model[['Empresa','Fecha']])]
      .reset_index()
)

# 5) Reconstruir X, y, groups y fechas desde el df_model ya filtrado
X = df_model[predictor_cols_final].copy().astype('float32')
y = df_model[TARGET_COL].copy()
groups = df_model['Empresa'].copy()
DATES_SERIES = pd.to_datetime(df_model['Fecha']).reset_index(drop=True)

# 6) Higiene final (por si acaso)
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(0.0, inplace=True)
mask_ok = y.notna().to_numpy() & np.isfinite(y.to_numpy())

X = X.loc[mask_ok].reset_index(drop=True)
y = y.loc[mask_ok].reset_index(drop=True)
groups = groups.loc[mask_ok].reset_index(drop=True)
DATES_SERIES = DATES_SERIES.loc[mask_ok].reset_index(drop=True)

# Alinear también los dataframes de compatibilidad con el mismo mask
df_model_sorted = df_model_sorted.loc[mask_ok].reset_index(drop=True)
df_sorted       = df_sorted.loc[mask_ok].reset_index(drop=True)

# 7) Reporte de diagnóstico
print(f"  · Meses totales: {n_by_month.shape[0]}  |  Meses válidos: {len(good_months)}")
bad_months = n_by_month.index[~n_by_month.index.isin(good_months)]
if len(bad_months):
    worst = n_by_month.loc[bad_months].sort_values().head(5)
    print("  · Ejemplos de meses filtrados (universo pequeño):")
    print(worst)

print(f"  · Dataset final tras filtro: X={X.shape}, y={y.shape}, empresas={groups.nunique()}")



🧹  Filtrando meses con universo insuficiente…
  · Meses totales: 113  |  Meses válidos: 113
  · Dataset final tras filtro: X=(20340, 36), y=(20340,), empresas=180


In [21]:
# ======================================================================
# GKXEnsemble — Ensamble simple de MLPs con interfaz scikit-learn
# ======================================================================
from sklearn.base import BaseEstimator, RegressorMixin
import numpy as np

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
except Exception as e:
    tf = None
    print("[GKXEnsemble] TensorFlow no disponible:", e)

class GKXEnsemble(BaseEstimator, RegressorMixin):
    def __init__(self,
                 n_ens=5,
                 hidden=(64, 32, 16),
                 dropout=0.2,
                 l2=1e-4,
                 lr=1e-3,
                 batch=256,
                 epochs=60,
                 patience=10,
                 val_frac=0.2,
                 verbose=0,
                 random_state=42):
        self.n_ens=n_ens; self.hidden=hidden; self.dropout=dropout
        self.l2=l2; self.lr=lr; self.batch=batch; self.epochs=epochs
        self.patience=patience; self.val_frac=val_frac; self.verbose=verbose
        self.random_state=random_state
        self._models=[]

    def _build_one(self, n_features, seed):
        keras.utils.set_random_seed(seed)
        reg = keras.regularizers.l2(self.l2) if self.l2 else None
        x = inputs = keras.Input(shape=(n_features,))
        for h in self.hidden:
            x = layers.Dense(h, activation="relu", kernel_regularizer=reg)(x)
            x = layers.BatchNormalization()(x)
            if self.dropout and self.dropout>0:
                x = layers.Dropout(self.dropout)(x)
        outputs = layers.Dense(1)(x)
        m = keras.Model(inputs, outputs)
        m.compile(optimizer=keras.optimizers.Adam(self.lr), loss="mse")
        return m

    def fit(self, X, y):
        if tf is None:
            raise RuntimeError("TensorFlow/Keras no disponible.")
        X = np.asarray(X, np.float32)
        y = np.asarray(y, np.float32).reshape(-1,1)
        n = X.shape[0]; n_val = int(max(1, np.floor(self.val_frac*n)))
        X_tr, X_val = (X[:-n_val], X[-n_val:]) if n_val>0 else (X, X[:0])
        y_tr, y_val = (y[:-n_val], y[-n_val:]) if n_val>0 else (y, y[:0])

        self._models=[]; base_seed=int(self.random_state)
        cbs=[]
        if n_val>0:
            cbs=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=self.patience,
                                               restore_best_weights=True, verbose=0),
                 keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=max(3,self.patience//2),
                                                   factor=0.5, min_lr=1e-5, verbose=0)]
        for i in range(self.n_ens):
            m = self._build_one(X.shape[1], seed=base_seed+i)
            m.fit(X_tr, y_tr, validation_data=(X_val, y_val) if n_val>0 else None,
                  batch_size=self.batch, epochs=self.epochs, verbose=self.verbose, callbacks=cbs)
            self._models.append(m)
        return self

    def predict(self, X):
        X = np.asarray(X, np.float32)
        if not self._models:
            return np.zeros(X.shape[0], dtype=np.float32)
        preds = [m.predict(X, verbose=0).reshape(-1) for m in self._models]
        return np.mean(preds, axis=0).astype(np.float32)


In [22]:
# ======================================================================
# SETUP FASE 1 — csZ en todos los modelos + XGBoost en GPU (LGBM en CPU)
# ======================================================================

import os, gc, time, warnings, joblib, numpy as np, pandas as pd
warnings.filterwarnings("ignore")

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import mean_squared_error, r2_score, make_scorer
from scipy.stats import spearmanr, loguniform, uniform

# Modelos
from sklearn.linear_model import ElasticNet, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.neural_network import MLPRegressor

# ========= Config general (tuneable) =========
FAST_MODE      = False
GPU            = True     # ← XGBoost en GPU
GPU_LGBM       = False    # ← dejá LGBM en CPU salvo que tengas build con GPU
CPU_CORES      = os.cpu_count() or 4

# Hilos para modelos CPU (evita calentar demasiado la máquina)
N_JOBS         = max(1, min(4, CPU_CORES - 1))

# Hilos para RandomizedSearchCV (1 si usás GPU para no competir)
SEARCH_N_JOBS  = 1

TRAIN_MIN = 24
GAP       = 1
HOLD      = 1
MAX_FOLDS = None
N_ITER_RS = 20 if not FAST_MODE else 8
TOP_N     = 1

SAVE_DIR  = "best_models_phase1"
SAVE_OOF  = True
OOF_DIR   = "artifacts/oof_phase1"
RUN_TAG   = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

ALLOW_GKX = True

# Qué modelos tunear
TUNE_MODELS = {"ElasticNet", "Random Forest", "XGBoost", "LightGBM", "MLP clásico"}

# ===== Scorer Spearman IC =====
def _spearman_ic(y_true, y_pred):
    ic = spearmanr(y_true, y_pred)[0]
    return 0.0 if (ic is None or np.isnan(ic)) else float(ic)
spearman_scorer = make_scorer(_spearman_ic, greater_is_better=True)

# ===== LightGBM kwargs (CPU por defecto) =====
def _lgbm_kwargs(GPU=False, FAST_MODE=False, N_JOBS=1):
    base = dict(
        random_state=42, n_estimators=(200 if FAST_MODE else 400),
        n_jobs=N_JOBS, verbose=-1,
        num_leaves=31, min_data_in_leaf=50,
        feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
        lambda_l2=2.0, max_bin=255, learning_rate=0.05,
        force_col_wise=True
    )
    try:
        base['device_type'] = ('gpu' if GPU else 'cpu')
    except TypeError:
        base['device'] = ('gpu' if GPU else 'cpu')
    return base

# ===== Construcción de pipelines (TODOS con csZ) =====
def build_models_and_search(FAST_MODE=False, GPU=True, GPU_LGBM=False, N_JOBS=1, ALLOW_GKX=False):
    # XGBoost en GPU (solo árbol), CPU mínima
    xgb_kwargs = dict(
        random_state=42,
        n_estimators=(200 if FAST_MODE else 400),
        verbosity=0,
        n_jobs=1,                              # la GPU hace el trabajo
        tree_method=('gpu_hist' if GPU else 'hist'),
        predictor=('gpu_predictor' if GPU else 'auto'),
        subsample=0.8, colsample_bytree=0.8
    )
    lgbm_kwargs = _lgbm_kwargs(GPU=GPU_LGBM, FAST_MODE=FAST_MODE, N_JOBS=N_JOBS)

    models_to_compare = {
        'Lineales': {
            'OLS (Linear Regression)': Pipeline([
                ('winsor_cs', winsor_cs),
                ('cs_z', zscore_cs),
                ('nanfix', nan_fix),
                ('model', LinearRegression())
            ]),
            'ElasticNet': Pipeline([
                ('winsor_cs', winsor_cs),
                ('cs_z', zscore_cs),
                ('nanfix', nan_fix),
                ('model', ElasticNet(random_state=42, max_iter=10000))
            ]),
        },
        'Boosting': {
            'Random Forest': Pipeline([
                ('winsor_cs', winsor_cs),
                ('cs_z', zscore_cs),
                ('nanfix', nan_fix),
                ('model', RandomForestRegressor(
                    random_state=42, n_jobs=N_JOBS,
                    n_estimators=(500 if not FAST_MODE else 200),
                    max_depth=None, min_samples_split=5, min_samples_leaf=2, max_features='sqrt'
                ))
            ]),
            'XGBoost': Pipeline([
                ('winsor_cs', winsor_cs),
                ('cs_z', zscore_cs),
                ('nanfix', nan_fix),
                ('model', XGBRegressor(**xgb_kwargs))
            ]),
            'LightGBM': Pipeline([   # versión csZ
                ('winsor_cs', winsor_cs),
                ('cs_z', zscore_cs),
                ('nanfix', nan_fix),
                ('model', LGBMRegressor(**lgbm_kwargs))
            ]),
        },
        'Redes neuronales': {
            'MLP clásico': Pipeline([
                ('winsor_cs', winsor_cs),
                ('cs_z', zscore_cs),
                ('nanfix', nan_fix),
                ('model', MLPRegressor(
                    random_state=42,
                    max_iter=(400 if not FAST_MODE else 200),
                    early_stopping=True, n_iter_no_change=10
                ))
            ]),
            **({
                'GKX NN (64-32-16)': Pipeline([
                    ('winsor_cs', winsor_cs),
                    ('cs_z', zscore_cs),
                    ('nanfix', nan_fix),
                    ('model', GKXEnsemble(
                        n_ens=(5 if not FAST_MODE else 3),
                        hidden=(64,32,16),
                        dropout=0.2, l2=1e-4,
                        lr=1e-3, batch=256,
                        epochs=(60 if not FAST_MODE else 40),
                        patience=10, verbose=0, random_state=42
                    ))
                ])
            } if ALLOW_GKX and 'GKXEnsemble' in globals() else {})
        }
    }

    # Grids de búsqueda
    param_dist_en = {'model__alpha': loguniform(1e-4, 1e1), 'model__l1_ratio': uniform(0.0, 1.0)}
    param_dist_rf = {'model__max_depth': [10, 20, None]}
    param_dist_xgb = {'model__max_depth': [3, 5, 7], 'model__learning_rate': [0.01, 0.05, 0.1],
                      'model__subsample': [0.7, 0.9], 'model__colsample_bytree': [0.7, 0.9]}
    param_dist_lgbm = {
        'model__max_depth': [3, 5, -1],
        'model__learning_rate': [0.01, 0.03, 0.05, 0.1],
        'model__num_leaves': [31, 63, 127],
        'model__min_data_in_leaf': [25, 50, 100],
        'model__feature_fraction': [0.7, 0.85, 1.0],
        'model__bagging_fraction': [0.7, 0.9, 1.0],
        'model__lambda_l2': [0.0, 1.0, 5.0],
    }
    param_dist_mlp = {'model__hidden_layer_sizes': [(50,), (100,), (50, 25)],
                      'model__activation': ['relu', 'tanh'],
                      'model__alpha': [1e-4, 1e-3, 1e-2],
                      'model__learning_rate_init': [1e-3, 5e-3]}

    search_configs = {
        'ElasticNet': param_dist_en,
        'Random Forest': param_dist_rf,
        'XGBoost': param_dist_xgb,
        'LightGBM': param_dist_lgbm,
        'MLP clásico': param_dist_mlp
    }
    return models_to_compare, search_configs

print("✅ Pipelines listos: csZ en todos; XGB en GPU; LGBM en CPU (por ahora).")

✅ Pipelines listos: csZ en todos; XGB en GPU; LGBM en CPU (por ahora).


In [23]:
# ===============================================================================
# BLOQUE · Transformadores Cross-Sectional (versión corregida INDEX-SAFE)
#   - No resetean índices en transform()
#   - Alinean por etiquetas del subset (train/test) para evitar misalign
#   - Devuelven np.ndarray float32 (compat pipelines)
# ===============================================================================

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer

class CrossSectionalZ(BaseEstimator, TransformerMixin):
    """
    Z-score cross-sectional por período (mes por defecto).
    - dates_full: Serie/array con TODAS las fechas del dataset (mismo orden posicional que X completo).
    - En transform(), X puede ser un subset (train/test) con índice de filas del original (RangeIndex).
      Tomamos ese índice como posiciones para mapear períodos desde dates_full, sin resetear.
    """
    def __init__(self, dates_full, freq='M', fill_invalid='zero', print_debug=False):
        self.dates_full = dates_full
        self.freq = freq
        self.fill_invalid = fill_invalid  # 'zero' o 'error'
        self.print_debug = print_debug

    def _prepare_full(self):
        s = pd.to_datetime(pd.Series(self.dates_full), errors='coerce')
        self.dates_full_ = s
        self.per_full_ = s.dt.to_period(self.freq)  # Series de Period[M]

    def fit(self, X, y=None):
        self._prepare_full()
        return self

    def transform(self, X, y=None, dates=None):
        Xdf = pd.DataFrame(X).astype('float32', copy=False)
        n_rows, n_cols = Xdf.shape
        if n_rows == 0 or n_cols == 0:
            raise ValueError("CrossSectionalZ: entrada vacía (0 filas o 0 columnas).")

        idx_sub = Xdf.index  # etiquetas del subset (deberían venir del RangeIndex original)

        # períodos alineados a *posiciones* del subset, con el MISMO índice que Xdf
        if dates is None:
            if not hasattr(self, 'per_full_'):
                self._prepare_full()
            max_pos = len(self.per_full_) - 1
            pos = np.clip(idx_sub.to_numpy(), 0, max_pos)  # usa las etiquetas como posiciones
            per_arr = self.per_full_.iloc[pos].to_numpy()
        else:
            ds = pd.to_datetime(pd.Series(dates), errors='coerce')
            if len(ds) != len(Xdf):
                ds = ds.iloc[:len(Xdf)]
            per_arr = ds.dt.to_period(self.freq).to_numpy()

        per_s = pd.Series(per_arr, index=idx_sub)
        valid = per_s.notna().to_numpy()

        if self.print_debug:
            print(f"[CrossSectionalZ] rows={n_rows}, cols={n_cols}, valid_dates={valid.sum()}, invalid={n_rows-valid.sum()}")

        if valid.sum() == 0:
            if self.fill_invalid == 'error':
                raise ValueError("CrossSectionalZ: todas las fechas del subset son NaT.")
            return np.zeros((n_rows, n_cols), dtype='float32')

        Xv   = Xdf.loc[valid]
        perv = per_s.loc[valid]  # MISMO índice que Xv

        def _z(g):
            g = g.astype('float32')
            mu = g.mean(axis=0)
            sd = g.std(axis=0, ddof=0)
            sd = np.maximum(sd, 1e-6)
            return (g - mu) / sd

        Zv = (Xv.groupby(perv, group_keys=False)
                .apply(_z)
                .replace([np.inf, -np.inf], 0.0)
                .fillna(0.0)
                .astype('float32'))

        # armo salida y asigno por etiquetas (evita descalces)
        Zout = pd.DataFrame(0.0, index=idx_sub, columns=Xdf.columns, dtype='float32')
        Zout.loc[Xv.index] = Zv.to_numpy(dtype='float32', copy=False)
        return Zout.to_numpy(dtype='float32', copy=False)


class CrossSectionalWinsorizer(BaseEstimator, TransformerMixin):
    """
    Winsorización cross-sectional por período (mes por defecto).
    - dates_full: Serie/array con TODAS las fechas del dataset.
    - Alineación por índice del subset; SIN resetear índices.
    """
    def __init__(self, dates_full, freq='M', lower=0.01, upper=0.01, min_group=5, print_debug=False):
        self.dates_full  = dates_full
        self.freq        = freq
        self.lower       = float(lower)
        self.upper       = float(upper)
        self.min_group   = int(min_group)
        self.print_debug = print_debug

    def _prepare_full(self):
        s = pd.to_datetime(pd.Series(self.dates_full), errors='coerce')
        self.dates_full_ = s
        self.per_full_   = s.dt.to_period(self.freq)

    def fit(self, X, y=None):
        self._prepare_full()
        return self

    def transform(self, X, y=None, dates=None):
        Xdf = pd.DataFrame(X).astype('float32', copy=False)
        n_rows, n_cols = Xdf.shape
        if n_rows == 0 or n_cols == 0:
            raise ValueError("CrossSectionalWinsorizer: entrada vacía.")

        idx_sub = Xdf.index

        if dates is None:
            if not hasattr(self, 'per_full_'):
                self._prepare_full()
            max_pos = len(self.per_full_) - 1
            pos = np.clip(idx_sub.to_numpy(), 0, max_pos)
            per_arr = self.per_full_.iloc[pos].to_numpy()
        else:
            ds = pd.to_datetime(pd.Series(dates), errors='coerce')
            if len(ds) != len(Xdf):
                ds = ds.iloc[:len(Xdf)]
            per_arr = ds.dt.to_period(self.freq).to_numpy()

        per_s = pd.Series(per_arr, index=idx_sub)
        valid = per_s.notna().to_numpy()

        if self.print_debug:
            print(f"[CSWinsor] rows={n_rows}, cols={n_cols}, valid={valid.sum()}, invalid={n_rows-valid.sum()}")

        if valid.sum() == 0:
            return Xdf.to_numpy(dtype='float32', copy=False)

        Xv   = Xdf.loc[valid]
        perv = per_s.loc[valid]  # alineado

        def _winsor_group(g):
            if len(g) < self.min_group:
                return g
            try:
                qlow = g.quantile(self.lower, numeric_only=True)
                qhi  = g.quantile(1.0 - self.upper, numeric_only=True)
            except TypeError:
                qlow = g.quantile(self.lower)
                qhi  = g.quantile(1.0 - self.upper)
            return g.clip(lower=qlow, upper=qhi, axis=1)

        Wv = (Xv.groupby(perv, group_keys=False).apply(_winsor_group).astype('float32'))

        Xout = Xdf.copy()
        Xout.loc[Xv.index] = Wv.to_numpy(dtype='float32', copy=False)
        Xout = Xout.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype('float32')
        return Xout.to_numpy(dtype='float32', copy=False)


# ===============================================================================
# BLOQUE 11.y · Refresco tras filtro de universo (re-instanciar + rebuild)
#   - Reinstancia zscore_cs / winsor_cs con DATES_SERIES actual
#   - Refresca GROUPS_SERIES
#   - Reconstruye MODELS_ALL / SEARCH_CFG para que usen estas instancias
# ===============================================================================

# 1) Re-instanciar transformadores dependientes de fechas
zscore_cs = CrossSectionalZ(
    dates_full=DATES_SERIES,
    freq='M',
    fill_invalid='zero',
    print_debug=False
)

winsor_cs = CrossSectionalWinsorizer(
    dates_full=DATES_SERIES,
    freq='M',
    lower=0.01, upper=0.01,
    min_group=5,
    print_debug=False
)

# 2) Refrescar GROUPS_SERIES (inner GroupKFold y runner)
GROUPS_SERIES = groups.reset_index(drop=True)

# 3) Reconstruir pipelines con las NUEVAS instancias
MODELS_ALL, SEARCH_CFG = build_models_and_search(
    FAST_MODE=FAST_MODE,
    GPU=GPU,
    N_JOBS=N_JOBS,
    ALLOW_GKX=ALLOW_GKX
)

print("✅ Transformadores y pipelines refrescados (versión index-safe).")


✅ Transformadores y pipelines refrescados (versión index-safe).


In [25]:
# ======================================================================
# RUNNER FASE 1 — purged walk-forward + (opcional) tuning interno
# ======================================================================
import time, os, gc, joblib, numpy as np, pandas as pd
from copy import deepcopy
from sklearn.base import clone
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import spearmanr

def _iter_time_blocks(months, train_min=24, gap=1, hold=1):
    """Genera splits purged walk-forward sobre un vector de Period[M]."""
    uniq = np.array(sorted(pd.unique(months)))
    for j in range(train_min + gap, len(uniq) - hold + 1):
        m_test_start = uniq[j]
        m_test_end   = uniq[j + hold - 1]
        m_cut        = m_test_start - gap
        tr_idx = np.where(months <  m_cut)[0]
        te_idx = np.where((months >= m_test_start) & (months <= m_test_end))[0]
        if tr_idx.size and te_idx.size:
            yield tr_idx, te_idx

def _make_inner_purged_cv(months_tr, n_splits=3, gap=1, hold=1, min_train=12):
    """Crea folds internos purged dentro del bloque de entrenamiento."""
    uniq = np.array(sorted(pd.unique(months_tr)))
    if len(uniq) < (min_train + gap + hold + 1):
        return []
    # Puntos equiespaciados para ubicar validaciones
    starts = np.linspace(min_train + gap, len(uniq) - hold, num=n_splits, dtype=int)
    folds = []
    used = set()
    for j in starts:
        if j in used or j >= len(uniq) - hold + 1:
            continue
        used.add(j)
        m_val_start = uniq[j]
        m_val_end   = uniq[min(j + hold - 1, len(uniq) - 1)]
        m_cut       = m_val_start - gap
        tr_mask = months_tr < m_cut
        va_mask = (months_tr >= m_val_start) & (months_tr <= m_val_end)
        tr_idx = np.where(tr_mask)[0]; va_idx = np.where(va_mask)[0]
        if tr_idx.size and va_idx.size:
            folds.append((tr_idx, va_idx))
    # Fallback mínimo
    if not folds and len(uniq) >= (min_train + gap + hold + 1):
        m_val_start = uniq[-hold]; m_val_end = uniq[-1]; m_cut = m_val_start - gap
        tr_idx = np.where(months_tr < m_cut)[0]
        va_idx = np.where((months_tr >= m_val_start) & (months_tr <= m_val_end))[0]
        if tr_idx.size and va_idx.size:
            folds.append((tr_idx, va_idx))
    return folds

def run_phase1_for_families(
    families=None,
    only_models=None,
    save_oof=True,
    n_jobs=1,
    train_min=None, gap=None, hold=None
):
    """
    Ejecuta fase 1 para los modelos indicados (con pipelines ya csZ).
    Usa purged walk-forward para OOF y (si corresponde) RandomizedSearchCV interno.
    """
    # ====== globals requeridos del setup ======
    global X, y, groups, DATES_SERIES, MODELS_ALL, SEARCH_CFG
    global TRAIN_MIN, GAP, HOLD, N_ITER_RS, spearman_scorer
    global SAVE_DIR, OOF_DIR, RUN_TAG, TUNE_MODELS

    train_min = train_min or TRAIN_MIN
    gap       = gap or GAP
    hold      = hold or HOLD

    months = pd.to_datetime(DATES_SERIES).dt.to_period('M').to_numpy()

    # Selección de modelos por familia/nombre
    families = families or list(MODELS_ALL.keys())
    selected = []
    for fam in families:
        if fam not in MODELS_ALL: 
            continue
        for model_name, pipe in MODELS_ALL[fam].items():
            if (only_models is not None) and (model_name not in only_models):
                continue
            selected.append((fam, model_name, pipe))

    if not selected:
        raise ValueError("No hay modelos seleccionados para correr.")

    os.makedirs(SAVE_DIR, exist_ok=True)
    if save_oof:
        os.makedirs(OOF_DIR, exist_ok=True)

    rows = []
    best_models = {}

    for fam, model_name, base_pipe in selected:
        t0 = time.time()
        oof = np.full(X.shape[0], np.nan, dtype=float)
        n_splits = 0

        for tr_idx, te_idx in _iter_time_blocks(months, train_min, gap, hold):
            X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
            X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]

            est = clone(base_pipe)

            # Tuning interno opcional (purged dentro del bloque train)
            if (model_name in TUNE_MODELS) and (model_name in SEARCH_CFG) and len(X_tr) > 200:
                inner_months = months[tr_idx]
                inner_cv = _make_inner_purged_cv(inner_months, n_splits=3, gap=gap, hold=hold, min_train=max(12, train_min//2))
                if inner_cv:
                    search = RandomizedSearchCV(
                        estimator=est,
                        param_distributions=SEARCH_CFG[model_name],
                        n_iter=N_ITER_RS,
                        scoring=spearman_scorer,
                        cv=inner_cv,
                        n_jobs=n_jobs,
                        random_state=42,
                        verbose=0
                    )
                    est = search.fit(X_tr, y_tr).best_estimator_
                else:
                    est = est.fit(X_tr, y_tr)
            else:
                est = est.fit(X_tr, y_tr)

            oof[te_idx] = est.predict(X_te)
            n_splits += 1

        mask = ~np.isnan(oof)
        if mask.sum() == 0:
            sp, rm, r2v = np.nan, np.nan, np.nan
        else:
            sp = spearmanr(y.iloc[mask], oof[mask])[0]
            rm = mean_squared_error(y.iloc[mask], oof[mask], squared=False)
            r2v= r2_score(y.iloc[mask], oof[mask])

        # Guardados
        if save_oof:
            oof_df = pd.DataFrame({
                'Fecha': pd.to_datetime(DATES_SERIES.iloc[mask]).values,
                'Empresa': groups.iloc[mask].values,
                'y_true': y.iloc[mask].values,
                'y_pred': oof[mask]
            })
            oof_path = os.path.join(OOF_DIR, f"{RUN_TAG}_{fam}_{model_name.replace(' ','_')}_oof.csv")
            oof_df.to_csv(oof_path, index=False)

        # Re-entrenar con todo y guardar
        final_est = clone(base_pipe).fit(X, y)
        mdl_path = os.path.join(SAVE_DIR, f"{RUN_TAG}_{fam}_{model_name.replace(' ','_')}.joblib")
        joblib.dump(final_est, mdl_path)
        best_models[model_name] = {'path': mdl_path, 'estimator': final_est}

        rows.append({
            'Familia': fam,
            'Modelo': model_name,
            'Folds': n_splits,
            'Spearman': sp if np.isfinite(sp) else np.nan,
            'RMSE': rm if np.isfinite(rm) else np.nan,
            'R2': r2v if np.isfinite(r2v) else np.nan,
            'Tiempo_s': round(time.time() - t0, 2)
        })

        gc.collect()

    lb = pd.DataFrame(rows).sort_values('Spearman', ascending=False).reset_index(drop=True)
    return lb, best_models


In [26]:
# ===============================================================================
# BLOQUE: Corrida Secuencial Uno por Vez (para Liberar Memoria y Evitar Cuelgues)
# - Llama run_phase1 para cada modelo individual, guarda OOF/best, gc.collect().
# - Opcional: del vars post-run para extra free RAM.
# - Alineado GKX: Evalúa incremental, compara OOS por modelo antes de L-S full.
# ===============================================================================

lista_models = [
    ('Lineales', 'OLS (Linear Regression)'),
    ('Lineales', 'ElasticNet'),
    ('Boosting', 'Random Forest'),
    ('Boosting', 'XGBoost'),
    ('Boosting', 'LightGBM'),
    ('Redes neuronales', 'MLP clásico'),
    ('Redes neuronales', 'GKX NN (64-32-16)') if ALLOW_GKX else None  # Si enabled
]
lista_models = [m for m in lista_models if m]  # Limpia None

all_leaderboards = []
all_best_models = {}

for fam, mod in lista_models:
    print(f"\n=== Corriendo {fam} - {mod} (uno por vez) ===")
    df_lb, best_md = run_phase1_for_families(
        families=[fam],
        only_models=[mod],
        save_oof=True,
        n_jobs=SEARCH_N_JOBS  
    )
    all_leaderboards.append(df_lb)
    all_best_models.update(best_md)
    
    # Libera explícito
    gc.collect()
    # Opcional: Del vars temporales si OOM severo (e.g., large OOF)
    # del df_lb, best_md  # Si no necesitas en mem, recarga de disk post

full_leaderboard = pd.concat(all_leaderboards).sort_values('Spearman', ascending=False)
print("\nLeaderboard Final Todos Modelos:")
print(full_leaderboard)


=== Corriendo Lineales - OLS (Linear Regression) (uno por vez) ===

=== Corriendo Lineales - ElasticNet (uno por vez) ===

=== Corriendo Boosting - Random Forest (uno por vez) ===

=== Corriendo Boosting - XGBoost (uno por vez) ===

=== Corriendo Boosting - LightGBM (uno por vez) ===

=== Corriendo Redes neuronales - MLP clásico (uno por vez) ===

=== Corriendo Redes neuronales - GKX NN (64-32-16) (uno por vez) ===
INFO:tensorflow:Assets written to: ram://18ca1353-aef2-47ab-b0c4-ff71c8d61f7a/assets
INFO:tensorflow:Assets written to: ram://1d54103b-dd39-4969-a369-5077847d2ec7/assets
INFO:tensorflow:Assets written to: ram://3e2ddd1a-436f-4f78-8beb-056997a064bf/assets
INFO:tensorflow:Assets written to: ram://d004444d-cdf7-4d42-9151-27637757219a/assets
INFO:tensorflow:Assets written to: ram://f01dfeb7-a1de-4827-a1a6-5c3407c19f1f/assets

Leaderboard Final Todos Modelos:
            Familia                   Modelo  Folds  Spearman      RMSE  \
0  Redes neuronales        GKX NN (64-32-16)  

In [ ]:
full_leaderboard

,Familia,Modelo,Folds,Spearman,RMSE,R2,Tiempo_s
0,Redes neuronales,GKX NN (64-32-16),88,-0.002683,0.098200,-0.002277,11234.44
0,Boosting,XGBoost,88,-0.018803,0.101420,-0.069082,11502.07
0,Lineales,OLS (Linear Regression),88,-0.022112,0.098344,-0.005213,269.55
0,Redes neuronales,MLP clásico,88,-0.023873,0.101722,-0.075459,17232.64
0,Boosting,LightGBM,88,-0.025807,0.099945,-0.038220,7488.82
0,Boosting,Random Forest,88,-0.030705,0.099807,-0.035348,5746.80
0,Lineales,ElasticNet,88,-0.051286,0.098277,-0.003856,4970.04


**OOF (out-of-fold)**

* Cómo se obtiene: predicciones de CV (tus folds del purged walk-forward), concatenadas.
* Para qué sirve: **selección y diagnóstico** de modelos/hiperparámetros.
* Qué controla: evita “mirar el futuro” dentro de cada fold.
* Ojo: como lo usás para **elegir** el modelo, está sujeto a *selection bias* (lo “viste” durante el tuning). Es validación, no test final.

**OOS (out-of-sample)**

* Cómo se obtiene: período **completamente fuera** del proceso de selección (ej., los últimos N meses que no tocaste para elegir nada) **o** un walk-forward final con el modelo ya fijado.
* Para qué sirve: **evaluación final** y estimar performance “real” en producción.
* Qué controla: no hay fuga ni sobreajuste por elección, porque ese tramo no influyó en el tuning.

**Regla práctica**

* Usa **OOF** para comparar y elegir (IC, t-stats, deciles, etc.).
* Reporta y decide con **OOS** (mismo set de métricas, pero ya con el modelo congelado).

**Matiz en tu setup**

* Cada fold del purged walk-forward **parece OOS**, pero al juntarlos y usarlos para seleccionar, el conjunto resultante es **OOF**.
* Luego corrés un **walk-forward final** (o reservas un holdout temporal) para obtener el **OOS** “de verdad”.

TL;DR: **OOF = validación para elegir. OOS = test final para creer.**


In [ ]:
# ===============================================================================
# NEUTRALIZACIÓN POR SECTOR / TAMAÑO (OOS)
#   - IC mensual (mean, std, t)
#   - Spread D10–D1 (fallback a Q5–Q1 si hay pocos nombres)
#   - Versiones: sector-neutral, size-neutral y sector+size neutral
# ===============================================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

def _ic_monthly(df, score_col='y_pred'):
    def _ic(g):
        if len(g) < 3 or g[score_col].nunique() < 2:
            return np.nan
        return spearmanr(g['y_true'], g[score_col])[0]
    ic_m = df.groupby('Mes', sort=True).apply(_ic).dropna()
    n = len(ic_m)
    ic_mean = ic_m.mean() if n else np.nan
    ic_std  = ic_m.std(ddof=1) if n > 1 else np.nan
    ic_t    = ic_mean / (ic_std / np.sqrt(n)) if (n > 1 and ic_std > 0) else np.nan
    hit     = (ic_m > 0).mean() if n else np.nan
    return ic_m, {'IC_mean': ic_mean, 'IC_std': ic_std, 'IC_t': ic_t, 'IC_hit_rate': hit, 'n_months_IC': n}

def _spread_series(df, score_col='y_pred'):
    # D10–D1; fallback a Q5–Q1 si <10 nombres
    def _spread(g):
        r = g[score_col].rank(method='first')
        try:
            if len(g) >= 10:
                d = pd.qcut(r, 10, labels=np.arange(1,11))
                m = g.assign(dec=d.astype(int)).groupby('dec')['y_true'].mean()
                return float(m.get(10, np.nan) - m.get(1, np.nan))
            elif len(g) >= 5:
                q = pd.qcut(r, 5, labels=np.arange(1,6))
                m = g.assign(q=q.astype(int)).groupby('q')['y_true'].mean()
                return float(m.get(5, np.nan) - m.get(1, np.nan))
            else:
                return np.nan
        except ValueError:
            return np.nan
    s = df.groupby('Mes', sort=True).apply(_spread).dropna()
    n = len(s)
    mean = s.mean() if n else np.nan
    std  = s.std(ddof=1) if n > 1 else np.nan
    tval = mean / (std / np.sqrt(n)) if (n > 1 and std > 0) else np.nan
    return s, {'Spread_mean': mean, 'Spread_t': tval, 'n_months_Spread': n}

def _z_within_group(g, col):
    s = g[col]
    mu, sd = s.mean(), s.std(ddof=0)
    return (s - mu) / (sd + 1e-9)

def _neutralize(oos_df, by_cols, base_col='y_pred', out_col='score_neut', method='zscore'):
    """Neutraliza y_pred por columnas en by_cols (por MES y luego por grupos)."""
    if isinstance(by_cols, str):
        by_cols = [by_cols]
    df = oos_df.copy()
    # Chequeos
    for c in by_cols:
        if c not in df.columns:
            raise ValueError(f"Falta la columna '{c}' en oos_df para neutralizar.")
    # z-score dentro de cada grupo (Mes, by_cols)
    if method == 'zscore':
        df[out_col] = (
            df.groupby(['Mes'] + by_cols, group_keys=False)
              .apply(lambda g: _z_within_group(g, base_col))
              .astype('float32')
        )
    elif method == 'demean':
        df[out_col] = (
            df[base_col] - df.groupby(['Mes'] + by_cols)[base_col].transform('mean')
        ).astype('float32')
    else:
        raise ValueError("method debe ser 'zscore' o 'demean'")
    return df

def metrics_neutralized(oos_df, kind='sector', method='zscore'):
    """
    kind: 'sector' | 'size' | 'sector_size'
    method: 'zscore' o 'demean'
    """
    assert {'Fecha','Mes','y_true','y_pred'}.issubset(oos_df.columns), "oos_df incompleto."
    df = oos_df.copy()

    if kind == 'sector':
        df = _neutralize(df, by_cols=['Sector_GICS'], base_col='y_pred', out_col='score_neut', method=method)
    elif kind == 'size':
        df = _neutralize(df, by_cols=['size_bucket'], base_col='y_pred', out_col='score_neut', method=method)
    elif kind == 'sector_size':
        df = _neutralize(df, by_cols=['Sector_GICS','size_bucket'], base_col='y_pred', out_col='score_neut', method=method)
    else:
        raise ValueError("kind inválido. Usa 'sector', 'size' o 'sector_size'.")

    # IC y Spread usando el score neutralizado
    ic_m, ic_sum = _ic_monthly(df, score_col='score_neut')
    spr_m, spr_sum = _spread_series(df, score_col='score_neut')

    summary = {**ic_sum, **spr_sum}
    return df, ic_m, spr_m, summary


In [ ]:
# ================================================================================
# BLOQUE: Análisis de Robustez OOS de los Top Modelos (post Fase 1)
# ================================================================================

print("\n" + "="*80)
print("### INICIO ANÁLISIS DE ROBUSTEZ OOS DE LOS TOP MODELOS ###")
print("="*80)

# --- Aplanar diccionario de modelos para acceso rápido ---
models_to_compare_flat = {model_name: pipe 
                          for family, family_dict in MODELS_ALL.items()
                          for model_name, pipe in family_dict.items()}

# --- Seleccionar los 3 mejores modelos del leaderboard ---
try:
    top_model_names = (
        df_leaderboard.sort_values('Spearman', ascending=False)
                      .head(3)['Modelo'].tolist()
    )
    print(f"Analizando la robustez de los 3 mejores modelos: {top_model_names}")
except (NameError, KeyError) as e:
    print(f"Error: No se pudo obtener 'df_leaderboard' o columna 'Modelo'. {e}")
    top_model_names = []

# --- Evaluación fold a fold ---
robustness_results = []
if top_model_names:
    start_time_robustness = time.time()

    for model_name in top_model_names:
        pipeline = models_to_compare_flat.get(model_name)
        if pipeline is None:
            print(f"⚠️  No se encontró el pipeline para '{model_name}'. Saltando.")
            continue

        print(f"\n--- Analizando consistencia de: {model_name} ---")
        start_time_model = time.time()

        ic_per_fold = []
        spread_per_fold = []
        y_pred_oof = np.full(y.shape[0], np.nan, dtype=float)

        for i, (train_idx, test_idx) in enumerate(cv_outer.split(X, y, groups=groups), 1):
            X_train_fold, X_test_fold = X.iloc[train_idx], X.iloc[test_idx]
            y_train_fold, y_test_fold = y.iloc[train_idx], y.iloc[test_idx]

            if model_name in search_configs:
                # Mantener esquema temporal y métrica Spearman
                inner_cv = TimeSeriesSplit(n_splits=3, gap=1)
                search = RandomizedSearchCV(
                    estimator=pipeline,
                    param_distributions=search_configs[model_name],
                    n_iter=10,
                    scoring=spearman_scorer,
                    cv=inner_cv,
                    n_jobs=-1,
                    random_state=42,
                    verbose=0
                )
                best_model = search.fit(X_train_fold, y_train_fold).best_estimator_
            else:
                best_model = clone(pipeline).fit(X_train_fold, y_train_fold)

            fold_preds = best_model.predict(X_test_fold)
            y_pred_oof[test_idx] = fold_preds

            ic_fold = spearmanr(y_test_fold, fold_preds)[0]
            ic_per_fold.append(ic_fold)

            # Spread D20–D80
            ranks = pd.Series(fold_preds, index=y_test_fold.index).rank(pct=True)
            long_returns = y_test_fold[ranks >= 0.80].mean()
            short_returns = y_test_fold[ranks <= 0.20].mean()
            spread_per_fold.append(long_returns - short_returns)

        ic_series = pd.Series(ic_per_fold)
        ic_mean = ic_series.mean()
        ic_std = ic_series.std()
        ic_t = ic_mean / (ic_std / np.sqrt(len(ic_series))) if ic_std > 0 else np.nan

        spread_series = pd.Series(spread_per_fold)
        sharpe_folds = spread_series.mean() / spread_series.std() if spread_series.std() > 0 else np.nan
        hit_rate = (spread_series > 0).mean()

        spearman_oof = spearmanr(y, y_pred_oof)[0]

        robustness_results.append({
            'Modelo': model_name,
            'Spearman_OOF': spearman_oof,
            'IC_Mean_Fold': ic_mean,
            'IC_Std_Fold': ic_std,
            'IC_t_Statistic': ic_t,
            'Sharpe_of_Folds': sharpe_folds,
            'Hit_Rate_Folds': hit_rate,
            'Tiempo (s)': time.time() - start_time_model
        })

    # --- Mostrar tabla ---
    df_robustness = pd.DataFrame(robustness_results).sort_values('IC_t_Statistic', ascending=False)

    print("\n" + "="*80)
    print("### RESULTADOS DE ROBUSTEZ OOS (ordenados por IC_t_Statistic) ###")
    print("="*80)
    display(df_robustness.style.format({
        'Spearman_OOF': '{:.4f}', 'IC_Mean_Fold': '{:.4f}', 'IC_Std_Fold': '{:.4f}',
        'IC_t_Statistic': '{:.2f}', 'Sharpe_of_Folds': '{:.2f}',
        'Hit_Rate_Folds': '{:.2%}', 'Tiempo (s)': '{:.2f}'
    }))

    print(f"\nTiempo total del análisis de robustez: {(time.time() - start_time_robustness)/60:.2f} minutos")
else:
    print("No hay modelos seleccionados para el análisis de robustez.")


In [1]:
# ================================================================================
# CHEQUEO 1: TEST DE PERMUTACIÓN (SANITY CHECK TEMPORAL · PURGED)
# ================================================================================

import numpy as np
from copy import deepcopy
from scipy.stats import spearmanr
import pandas as pd

# 0) Asegurar meses y modelo
months = pd.to_datetime(df_model['Fecha']).dt.to_period('M')

models_to_compare_flat = {mn: pipe
                          for fam, d in MODELS_ALL.items()
                          for mn, pipe in d.items()}
model_to_test = models_to_compare_flat['LightGBM']

# 1) Permutación con walk-forward mensual + embargo
def permutation_ic_purged(model, X, y, months, n_perm=50, train_min=24, gap=1, hold=1, seed=42):
    rng = np.random.RandomState(seed)
    y_np = y.to_numpy().astype(float)
    ic_scores = []

    # derivar uniq_m dentro de la función (auto-contenido)
    uniq_m = np.array(sorted(pd.unique(months)))

    for _ in range(n_perm):
        y_perm = y_np.copy()
        rng.shuffle(y_perm)

        ics = []
        for j in range(train_min + gap, len(uniq_m) - hold + 1):
            m_test_start = uniq_m[j]
            m_test_end   = uniq_m[j + hold - 1]
            m_cut        = m_test_start - gap

            tr_idx = (months < m_cut).values
            te_idx = ((months >= m_test_start) & (months <= m_test_end)).values

            if tr_idx.sum() == 0 or te_idx.sum() == 0:
                continue

            m = deepcopy(model).fit(X.loc[tr_idx], y_perm[tr_idx])
            preds = m.predict(X.loc[te_idx])
            ics.append(spearmanr(y_perm[te_idx], preds)[0])

        ic_scores.append(np.nanmean(ics))

    ic_scores = np.asarray(ic_scores, float)
    print(f"IC permutado — media: {np.nanmean(ic_scores):.4f} | std: {np.nanstd(ic_scores):.4f}")
    return ic_scores

print("\n--- 1. Ejecutando Test de Permutación (temporal · purged) ---")
perm_ic = permutation_ic_purged(model_to_test, X, y, months, n_perm=50)

perm_mean = float(np.nanmean(perm_ic))
print("✅ PASA: El modelo no encuentra señal en el ruido."
      if (np.isfinite(perm_mean) and abs(perm_mean) < 0.01)
      else "❌ FALLA: El modelo parece encontrar señal espuria.")


NameError: name 'df_model' is not defined

¡Buenísimo! Ese IC permutado ≈ –0.0018 (≈ 0) confirma que el modelo no está capturando señal espuria en el target ‒ al permutar y romper la relación cronológica, la correlación desaparece. ✔

Qué significa

Sin fuga de información en la tubería: si hubiera leak, el modelo habría conservado algo de predictibilidad y verías un IC sustancialmente > 0.1.

Distribución centrada en 0 (con alguna varianza pequeña) es justo el patrón esperado por puro ruido.

In [ ]:
# --- 2. Ejecutando Test de Lag Adicional (versión corregida con Period[M]) ---
from copy import deepcopy
import numpy as np
from scipy.stats import spearmanr
from lightgbm import LGBMRegressor
from sklearn.base import clone

print("\n--- 2. Ejecutando Test de Lag Adicional (versión corregida) ---")

# 0) Modelo a usar
model_to_test = models_to_compare_flat.get(
    'LightGBM',
    LGBMRegressor(random_state=42, n_jobs=-1)
)

# 1) Base de datos
X_base = df_model[predictor_cols_final].copy()
y_base = df_model[TARGET_COL].copy()
dates  = pd.to_datetime(df_model['Fecha'])
months = dates.dt.to_period('M')   # <<< Opción A: conservar Period[M]
groups = df_model['Empresa']

# 2) Aplicar un lag +1 a TODAS las features por empresa (lag adicional)
X_lagged = X_base.copy()
for c in X_lagged.columns:
    X_lagged[c] = X_lagged.groupby(groups)[c].shift(1)

# Opcional: limpiar filas sin datos tras el lag
mask_valid = ~X_lagged.isna().any(axis=1) & y_base.notna()
X_lagged = X_lagged.loc[mask_valid]
y_lagged = y_base.loc[mask_valid]
months_l = months.loc[mask_valid]

# 3) Construir splits temporales "purged"
TRAIN_MIN = 24   # meses mínimos de entrenamiento
GAP       = 1    # embargo en meses entre train y test
HOLD      = 1    # tamaño del bloque de test (meses)

uniq_m = np.array(sorted(pd.unique(months_l)))  # Period[M] ordenable

ics = []
for j in range(TRAIN_MIN + GAP, len(uniq_m) - HOLD + 1):
    # primer mes del test = uniq_m[j]
    m_test_start = uniq_m[j]
    m_test_end   = uniq_m[j + HOLD - 1]

    # Train: todo < (m_test_start - GAP)
    # Con Period[M], restar GAP meses:
    m_cut = m_test_start - GAP
    tr_idx = months_l < m_cut

    # Test: [m_test_start, m_test_end]
    te_idx = (months_l >= m_test_start) & (months_l <= m_test_end)

    if tr_idx.sum() == 0 or te_idx.sum() == 0:
        continue

    m = clone(model_to_test).fit(X_lagged.loc[tr_idx], y_lagged.loc[tr_idx])
    preds = m.predict(X_lagged.loc[te_idx])
    ic = spearmanr(y_lagged.loc[te_idx], preds)[0]
    ics.append(ic)

# 4) Resultado
ic_mean = float(np.nanmean(ics)) if len(ics) else np.nan
print(f"Resultado: IC con lag +1 (temporal, purged) = {ic_mean:.4f}")

if np.isnan(ic_mean) or abs(ic_mean) < 0.05:
    print("✅ PASA: el poder predictivo se diluye con un lag adicional (no hay fuga clara).")
else:
    print("❌ FALLA: la señal persiste con un lag adicional; posible leakage aún presente.")


In [ ]:
from copy import deepcopy
from scipy.stats import spearmanr
import numpy as np

# 1) Selecciona el modelo que quieras testear (rápido y representativo)
model_to_test = models_to_compare_flat['LightGBM']   # o 'XGBoost', etc.

def ic_with_extra_lag(k):
    """IC OOF aplicando un lag adicional de k meses a TODAS las features."""
    X_lag = X.copy()
    for col in X.columns:
        X_lag[col] = X.groupby(df_model['Empresa'])[col].shift(k)
    X_lag.fillna(0.0, inplace=True)        # imputación rápida

    ic_folds = []
    for tr, te in cv_outer.split(X_lag, y, groups):
        mdl = deepcopy(model_to_test).fit(X_lag.iloc[tr], y.iloc[tr])
        preds = mdl.predict(X_lag.iloc[te])
        ic_folds.append(spearmanr(preds, y.iloc[te])[0])
    return np.mean(ic_folds)

for k in range(0, 5):          # prueba lag +0 a +4
    print(f"Lag +{k} → IC = {ic_with_extra_lag(k):.4f}")


In [ ]:
from scipy.stats import spearmanr
import pandas as pd

probe = []
tgt = df_model[TARGET_COL]

for col in X.columns:
    # correlación original
    ic_now, _   = spearmanr(X[col], tgt)
    # correlación tras +1 mes extra
    shifted      = X.groupby(df_model['Empresa'])[col].shift(1)
    ic_shift, _ = spearmanr(shifted, tgt)
    probe.append((col, ic_now, ic_shift))

cols_sospechosas = (
    pd.DataFrame(probe, columns=['feature','IC_now','IC_shift'])
      .query('abs(IC_shift) > 0.05')        # umbral debatible
      .sort_values('IC_now', ascending=False)
)
display(cols_sospechosas.head(15))


In [ ]:
from scipy.stats import spearmanr
import pandas as pd
import numpy as np

tgt = df_model.loc[X.index, TARGET_COL]          # mismo índice que X

rows = []
emp  = df_model.loc[X.index, 'Empresa']          # vector de empresas alineado

for col in X.columns:
    col_now   = X[col]
    col_shift = col_now.groupby(emp).shift(1)    # desplaza dentro de cada empresa

    # si todo NaN → ic_shift = 0.0 para que cuente
    ic_now,   _ = spearmanr(col_now,   tgt)
    ic_shift, _ = spearmanr(col_shift.fillna(0), tgt)

    rows.append((col, ic_now, ic_shift))

probe_df = (pd.DataFrame(rows, columns=['feature','IC_now','IC_shift'])
              .assign(abs_shift=lambda d: d.IC_shift.abs())
              .sort_values('abs_shift', ascending=False))

display(probe_df.head(20))


In [ ]:
# ================================
# CHEQUEO 2: IC rolling OOS + Decile Spread
# ================================
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.base import clone
from lightgbm import LGBMRegressor

# --- 0) Modelo ---
model = models_to_compare_flat.get('LightGBM', LGBMRegressor(random_state=42, n_jobs=-1))

# --- 1) Datos base ---
X = df_model[predictor_cols_final].copy()
y = df_model[TARGET_COL].copy()
dates = pd.to_datetime(df_model['Fecha'])
months = dates.dt.to_period('M')        # Period[M] evita problemas de casting
empresas = df_model['Empresa']
has_sector = 'Sector' in df_model.columns
sectores = df_model['Sector'] if has_sector else None

# (opcional) quitar espacios en nombres para evitar warnings
X.columns = X.columns.str.replace(r'\s+', '_', regex=True)

# --- 2) Rolling OOS con embargo ("purged") ---
TRAIN_MIN = 24  # meses mínimos de train
GAP       = 1   # meses de embargo
HOLD      = 1   # tamaño test en meses

uniq_m = np.array(sorted(pd.unique(months)))  # Period[M]

oos_rows = []
for j in range(TRAIN_MIN + GAP, len(uniq_m) - HOLD + 1):
    m_test_start = uniq_m[j]
    m_test_end   = uniq_m[j + HOLD - 1]
    m_cut        = m_test_start - GAP

    tr_idx = months < m_cut
    te_idx = (months >= m_test_start) & (months <= m_test_end)

    if tr_idx.sum() == 0 or te_idx.sum() == 0:
        continue

    m = clone(model).fit(X.loc[tr_idx], y.loc[tr_idx])
    preds = m.predict(X.loc[te_idx])

    part = pd.DataFrame({
        'Fecha'  : dates.loc[te_idx].values,
        'Mes'    : months.loc[te_idx].values,
        'Empresa': empresas.loc[te_idx].values,
        'y_true' : y.loc[te_idx].values,
        'y_pred' : preds
    })
    if has_sector:
        part['Sector'] = sectores.loc[te_idx].values
    oos_rows.append(part)

oos = pd.concat(oos_rows, ignore_index=True) if oos_rows else pd.DataFrame()
print(f"OOS preds: {oos.shape}")

# --- 3) IC (Spearman) mensual OOS ---
def _safe_ic(g):
    if len(g) < 3 or g['y_pred'].nunique() < 2:
        return np.nan
    return spearmanr(g['y_true'], g['y_pred'])[0]

ic_month = oos.groupby('Mes', sort=True).apply(_safe_ic).rename('IC')
ic_month = ic_month.dropna()
n_m      = ic_month.shape[0]
ic_mean  = ic_month.mean()
ic_std   = ic_month.std(ddof=1)
ic_t     = ic_mean / ic_std * np.sqrt(n_m) if ic_std > 0 and n_m > 1 else np.nan
hit_rate = (ic_month > 0).mean()

print(f"IC mensual (OOS): mean={ic_mean:.4f}, std={ic_std:.4f}, t={ic_t:.2f}, hit-rate={hit_rate:.1%}, n={n_m}")

# --- 4) Decile spread OOS (equal-weight dentro de cada mes) ---
def _decile_spread(g):
    # rankeo para evitar empates raros en qcut
    r = g['y_pred'].rank(method='first')
    try:
        d = pd.qcut(r, 10, labels=np.arange(1,11))
    except ValueError:
        return np.nan
    g2 = g.copy()
    g2['dec'] = d.astype(int)
    mean_by_dec = g2.groupby('dec')['y_true'].mean()
    if 1 in mean_by_dec.index and 10 in mean_by_dec.index:
        return float(mean_by_dec.loc[10] - mean_by_dec.loc[1])
    return np.nan

spread_m = oos.groupby('Mes', sort=True).apply(_decile_spread).rename('D10_minus_D1')
spread_m = spread_m.dropna()
n_s      = spread_m.shape[0]
spr_mean = spread_m.mean()
spr_std  = spread_m.std(ddof=1)
spr_t    = spr_mean / spr_std * np.sqrt(n_s) if spr_std > 0 and n_s > 1 else np.nan

print(f"Decile spread (OOS): mean={spr_mean:.4f}, std={spr_std:.4f}, t={spr_t:.2f}, n={n_s}")

# --- 5) (Opcional) Decile spread sector-neutral por mes ---
if has_sector:
    def _sector_neutral_spread(gm):
        # rankeo dentro de sector, luego juntamos y formamos deciles globales con ese score neutralizado
        def z_within_sector(h):
            s = h['y_pred']
            mu, sd = s.mean(), s.std(ddof=0)
            return (s - mu) / (sd + 1e-9)

        gm = gm.copy()
        gm['z_sec'] = gm.groupby('Sector', group_keys=False).apply(z_within_sector)

        r = gm['z_sec'].rank(method='first')
        try:
            d = pd.qcut(r, 10, labels=np.arange(1,11))
        except ValueError:
            return np.nan
        gm['dec'] = d.astype(int)
        mean_by_dec = gm.groupby('dec')['y_true'].mean()
        if 1 in mean_by_dec.index and 10 in mean_by_dec.index:
            return float(mean_by_dec.loc[10] - mean_by_dec.loc[1])
        return np.nan

    spread_sn = oos.groupby('Mes', sort=True).apply(_sector_neutral_spread).rename('D10-D1_SN').dropna()
    if len(spread_sn):
        sn_mean = spread_sn.mean(); sn_std = spread_sn.std(ddof=1)
        sn_t = sn_mean / sn_std * np.sqrt(len(spread_sn)) if sn_std > 0 and len(spread_sn) > 1 else np.nan
        print(f"Decile spread sector-neutral (OOS): mean={sn_mean:.4f}, std={sn_std:.4f}, t={sn_t:.2f}, n={len(spread_sn)}")


In [ ]:
# ================================================================================
# CHEQUEO 3: FORWARD-CHAIN CV (VALIDACIÓN TEMPORAL · GAP=1)
# ================================================================================

from sklearn.base import clone
from scipy.stats import spearmanr
import numpy as np
import pandas as pd

print("\n--- 3. Ejecutando Forward-Chain CV (gap=1) ---")

months = pd.to_datetime(df_model['Fecha']).dt.to_period('M')
uniq_m = np.array(sorted(pd.unique(months)))

models_to_compare_flat = {
    mn: pipe
    for fam, d in MODELS_ALL.items()
    for mn, pipe in d.items()
}
model_to_test = models_to_compare_flat['LightGBM']

TRAIN_MIN = 24
GAP       = 1
HOLD      = 1

ic_chain = []
for j in range(TRAIN_MIN + GAP, len(uniq_m) - HOLD + 1):
    m_test_start = uniq_m[j]
    m_test_end   = uniq_m[j + HOLD - 1]
    m_cut        = m_test_start - GAP

    tr_idx = months < m_cut
    te_idx = (months >= m_test_start) & (months <= m_test_end)

    if tr_idx.sum() == 0 or te_idx.sum() == 0:
        continue

    m = clone(model_to_test).fit(X.loc[tr_idx], y.loc[tr_idx])
    preds = m.predict(X.loc[te_idx])
    ic_chain.append(spearmanr(y.loc[te_idx], preds)[0])

# Resultado + veredicto robusto
if len(ic_chain) == 0:
    print("Resultado: no hubo splits válidos para calcular IC forward-chain.")
else:
    ic_fw = float(np.nanmean(ic_chain))
    print(f"Resultado: IC forward-chain = {ic_fw:.4f}")
    if np.isfinite(ic_fw) and ic_fw > 0.03:   # usa tu umbral global si lo definiste
        print("✅ PASA: señal con poder predictivo en validación temporal (gap=1).")
    else:
        print("❌ FALLA: señal insuficiente en forward-chain (gap=1).")


In [ ]:
# ================================================================================
# CHEQUEO 4: BACK-TEST “PURGED” POR MESES
# ================================================================================

from sklearn.base import clone
from scipy.stats import spearmanr
import numpy as np
import pandas as pd

print("\n--- 4. Ejecutando Back-test 'Purged' por meses ---")

models_to_compare_flat = {mn: pipe
                          for fam, d in MODELS_ALL.items()
                          for mn, pipe in d.items()}
model_to_test = models_to_compare_flat['LightGBM']

months = pd.to_datetime(df_model['Fecha']).dt.to_period('M')
uniq_m = np.array(sorted(pd.unique(months)))

GAP   = 3
HOLD  = 6
TRAIN = 24

ics, spreads = [], []
for j in range(TRAIN + GAP, len(uniq_m) - HOLD + 1):
    m_test_start = uniq_m[j]
    m_test_end   = uniq_m[j + HOLD - 1]
    m_cut        = m_test_start - GAP

    tr_idx = months < m_cut
    te_idx = (months >= m_test_start) & (months <= m_test_end)

    if tr_idx.sum() == 0 or te_idx.sum() == 0:
        continue

    mdl = clone(model_to_test).fit(X.loc[tr_idx], y.loc[tr_idx])
    preds = mdl.predict(X.loc[te_idx])

    ic = spearmanr(preds, y.loc[te_idx])[0]
    ics.append(ic)

    ranks = pd.Series(preds, index=y.loc[te_idx].index).rank(pct=True)
    long_ret  = y.loc[te_idx][ranks >= .80].mean()
    short_ret = y.loc[te_idx][ranks <= .20].mean()
    spreads.append(long_ret - short_ret)

if ics:
    ic_mean_purged  = float(np.mean(ics))
    sharpe_purged   = float(np.mean(spreads) / np.std(spreads, ddof=1)) if len(spreads) > 1 and np.std(spreads, ddof=1) > 0 else np.nan
    hit_rate_purged = float((pd.Series(ics) > 0).mean())

    print(f"\nIC medio purged = {ic_mean_purged:.3f}")
    print(f"Sharpe por bloques = {sharpe_purged:.2f}")
    print(f"Hit Rate (IC>0) = {hit_rate_purged:.2%}")

    IC_THRESHOLD = 0.05
    SHARPE_THRESHOLD = 0.5
    HIT_RATE_THRESHOLD = 0.60

    if (ic_mean_purged > IC_THRESHOLD) and (sharpe_purged > SHARPE_THRESHOLD) and (hit_rate_purged > HIT_RATE_THRESHOLD):
        print("\n✅ PASA: Señal robusta y consistente en validación temporal estricta.")
    else:
        print("\n❌ FALLA: Debilidades en validación temporal estricta.")
else:
    print("No se pudieron generar resultados.")


In [ ]:
# ================================================================================
# CHEQUEO 5: ESTABILIDAD POR TAMAÑO DE VENTANA
# ================================================================================

from sklearn.base import clone
from scipy.stats import spearmanr
import numpy as np
import pandas as pd

print("\n--- 5. Test de Estabilidad por Tamaño de Ventana (meses reales) ---")

models_to_compare_flat = {
    mn: pipe
    for fam, d in MODELS_ALL.items()
    for mn, pipe in d.items()
}
model_to_test = models_to_compare_flat['LightGBM']

months = pd.to_datetime(df_model['Fecha']).dt.to_period('M')
uniq_m = np.array(sorted(pd.unique(months)))

windows = [12, 24, 36, 48, 60]
test_w, gap = 6, 3

results = []
for win in windows:
    print(f"  Ventana train = {win}m...")
    ics, spreads = [], []
    for j in range(win + gap, len(uniq_m) - test_w + 1):
        m_test_start = uniq_m[j]
        m_test_end   = uniq_m[j + test_w - 1]
        m_cut        = m_test_start - gap

        tr_idx = (months >= uniq_m[j - gap - win]) & (months < m_cut)
        te_idx = (months >= m_test_start) & (months <= m_test_end)

        if tr_idx.sum() == 0 or te_idx.sum() == 0:
            continue

        mdl = clone(model_to_test).fit(X.loc[tr_idx], y.loc[tr_idx])
        preds = mdl.predict(X.loc[te_idx])

        ic = spearmanr(preds, y.loc[te_idx])[0]
        ics.append(ic)

        r = pd.Series(preds, index=y.loc[te_idx].index).rank(pct=True)
        l = y.loc[te_idx][r >= .80].mean()
        s = y.loc[te_idx][r <= .20].mean()
        spreads.append(l - s)

    if ics:
        ic_mean = float(np.mean(ics))
        ic_std  = float(np.std(ics, ddof=1)) if len(ics) > 1 else np.nan
        ic_t    = ic_mean / (ic_std/np.sqrt(len(ics))) if (len(ics) > 1 and ic_std > 0) else np.nan
        sharpe  = float(np.mean(spreads) / np.std(spreads, ddof=1)) if len(spreads) > 1 and np.std(spreads, ddof=1) > 0 else np.nan
        results.append({'train_window_m': win, 'IC_mean': ic_mean, 'IC_std': ic_std, 'IC_t': ic_t, 'Sharpe': sharpe})

if results:
    df_results_window = pd.DataFrame(results)
    print("\n--- Resultados del Test de Estabilidad por Ventana ---")
    display(df_results_window.style.format({
        'IC_mean': '{:.4f}',
        'IC_std': '{:.4f}',
        'IC_t': '{:.2f}',
        'Sharpe': '{:.2f}'
    }))

    # === Veredicto PASA/FALLA ===
    worst_ic    = df_results_window.loc[df_results_window['IC_mean'].idxmin()]
    worst_tstat = df_results_window.loc[df_results_window['IC_t'].idxmin()]

    all_ic_ok = (df_results_window['IC_mean'] > 0.03).all()      # IC_WINDOW_MIN
    all_t_ok  = (df_results_window['IC_t'] > 2.00).all()         # TSTAT_WINDOW_MIN

    if all_ic_ok and all_t_ok:
        print("✅ PASA: señal robusta y estable a distintos tamaños de ventana.")
    else:
        print("❌ FALLA: señal sensible al tamaño de ventana.")
        if not all_ic_ok:
            print(f"   - IC medio mínimo {worst_ic['IC_mean']:.4f} "
                  f"@ {worst_ic['train_window_m']}m ≤ 0.03")
        if not all_t_ok:
            print(f"   - t-stat mínimo {worst_tstat['IC_t']:.2f} "
                  f"@ {worst_tstat['train_window_m']}m ≤ 2.00")
else:
    print("No hubo splits válidos para esas ventanas.")


In [ ]:
# ================================================================================
# CHEQUEO 6: IC MENSUAL OOS + DRAWDOWN DE LA SEÑAL
# ================================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# ------------------------------------------------------------
# 1.  FUNCIÓN MEJORADA: calcula IC, métricas, grafica y devuelve un veredicto
# ------------------------------------------------------------
def analyze_ic_timeseries(oof_pred, y_true, dates,
                          freq: str = 'M',
                          min_obs: int = 2,
                          title: str = 'Cumulative IC (OOS)'):
    """
    Calcula y analiza la serie temporal del Information Coefficient (IC) OOS.
    Muestra un gráfico, imprime métricas clave y devuelve un veredicto.

    Returns
    -------
    dict: Un diccionario con las métricas calculadas y el veredicto.
    """
    print("\n--- 6. Analizando Estabilidad Temporal del IC (Out-of-Sample) ---")

    # --- 1. Construir y calcular la serie temporal de IC ---
    tmp = pd.DataFrame({'pred': oof_pred, 'target': y_true, 'date': pd.to_datetime(dates)}).dropna()
    tmp['grp'] = tmp['date'].dt.to_period(freq)
    ic_ts = (tmp.groupby('grp')
                .filter(lambda g: len(g) >= min_obs)
                .groupby('grp')
                .apply(lambda g: spearmanr(g['pred'], g['target'])[0])
                .astype(float)
                .sort_index())

    if ic_ts.empty:
        print("No se pudo calcular la serie temporal de IC (no hay suficientes datos).")
        return {'verdict': 'INCONCLUSO', 'ic_mean': np.nan, 'ic_t_stat': np.nan, 'drawdown': np.nan}

    # --- 2. Calcular métricas clave de la serie temporal ---
    ic_mean = ic_ts.mean()
    ic_std = ic_ts.std()
    ic_t_stat = ic_mean / (ic_std / np.sqrt(len(ic_ts))) if ic_std > 0 else np.inf
    
    cumulative_ic = ic_ts.cumsum()
    drawdown = (cumulative_ic - cumulative_ic.cummax()).min()
    
    # --- 3. Graficar el IC acumulado ---
    ax = cumulative_ic.plot(figsize=(12, 4), lw=2, title=title, ylabel='IC acumulado')
    ax.axhline(0, ls='--', c='grey', lw=0.8)
    plt.tight_layout()
    plt.show()

    # --- 4. Imprimir métricas y dar un veredicto ---
    print("\n--- Resultados del Análisis de Estabilidad del IC ---")
    print(f"  - IC Mensual Promedio: {ic_mean:.4f}")
    print(f"  - t-statistic del IC Mensual: {ic_t_stat:.2f}")
    print(f"  - Peor Drawdown del IC Acumulado: {drawdown:.2f}")

    # --- Lógica del Veredicto ---
    IC_MEAN_THRESHOLD = 0.03  # Un IC promedio > 3% es consistentemente bueno
    IC_T_STAT_THRESHOLD = 2.0   # Un t-stat > 2.0 es estadísticamente significativo
    DRAWDOWN_THRESHOLD = -1.0 # El drawdown no debe ser demasiado profundo

    ic_mean_pass = ic_mean > IC_MEAN_THRESHOLD
    t_stat_pass = ic_t_stat > IC_T_STAT_THRESHOLD
    drawdown_pass = drawdown > DRAWDOWN_THRESHOLD

    verdict = "✅ PASA" if ic_mean_pass and t_stat_pass and drawdown_pass else "❌ FALLA"
    
    print(f"\nVeredicto: {verdict}")
    if verdict == "❌ FALLA":
        if not ic_mean_pass:
            print(f"   - Razón: El IC medio ({ic_mean:.4f}) está por debajo del umbral de {IC_MEAN_THRESHOLD:.2f}.")
        if not t_stat_pass:
            print(f"   - Razón: El t-stat ({ic_t_stat:.2f}) está por debajo del umbral de {IC_T_STAT_THRESHOLD:.2f}.")
        if not drawdown_pass:
            print(f"   - Razón: El drawdown ({drawdown:.2f}) superó el umbral de {DRAWDOWN_THRESHOLD:.2f}.")
            
    return {
        'verdict': verdict,
        'ic_mean': ic_mean,
        'ic_t_stat': ic_t_stat,
        'drawdown': drawdown
    }

# ------------------------------------------------------------
# 2.  INVOCACIÓN DE LA FUNCIÓN MEJORADA
# ------------------------------------------------------------
# Asumimos que y_pred_oof, y, df_model existen de la Fase 1
# (y_pred_oof debe ser el array de predicciones del MEJOR modelo)

# Para asegurar que usamos las predicciones del mejor modelo (ej. XGBoost),
# podemos re-generar y_pred_oof si no estamos seguros de cuál contiene.
# (Esta parte es opcional si estás seguro de que y_pred_oof es el correcto)
print("Generando predicciones OOF para el mejor modelo (XGBoost) para el análisis...")
best_model_name = 'XGBoost' # O el que haya salido mejor en tu análisis de robustez
best_model_pipeline = models_to_compare_flat[best_model_name]
y_pred_oof_best = np.full_like(y, np.nan, dtype=float)
for tr, te in cv_outer.split(X, y, groups):
    mdl = clone(best_model_pipeline).fit(X.iloc[tr], y.iloc[tr])
    y_pred_oof_best[te] = mdl.predict(X.iloc[te])


# Llamar a la nueva función de análisis
ic_analysis_results = analyze_ic_timeseries(
    oof_pred=y_pred_oof_best,
    y_true=y,
    dates=df_model['Fecha']
)

### Kit completo de stress-tests para examinar la señal a fondo

In [ ]:
## 0. Setup común

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.base import clone

# -------- parámetros de uso general ----------
RANK_Q             = .20      # percentil para long/short
ANNUALIZE_FACTOR   = np.sqrt(12)  # si usas IC mensual
PLOT_STYLE         = dict(figsize=(10,4), lw=2)
# ---------------------------------------------


In [ ]:
# ================================================================================
# CHEQUEO 7: ROLLING IC DE 12 MESES + T-STAT DINÁMICO (con veredicto)
# ================================================================================

print("\n--- 7. Analizando Rolling IC y t-statistic ---")

# 1) Aplanar modelos si hace falta
if 'models_to_compare_flat' not in locals():
    models_to_compare_flat = {mn: pipe
                              for fam, d in models_to_compare.items()
                              for mn, pipe in d.items()}

# 2) Función de análisis (IN-SAMPLE)
from sklearn.base import clone
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def rolling_ic(model, window=12):
    # Entrena en todos los datos (IN-SAMPLE) para ver dinámica
    mdl = clone(model).fit(X, y)
    preds = pd.Series(mdl.predict(X), index=df_model.index)

    # IC mensual (agrupar por mes real)
    months = pd.to_datetime(df_model['Fecha']).dt.to_period('M')
    ic_by_month = preds.groupby(months).apply(lambda s: spearmanr(s, y.loc[s.index])[0]).astype(float)

    # Rolling IC y rolling t en ventana 'window'
    roll_ic = ic_by_month.rolling(window).mean()
    roll_std = ic_by_month.rolling(window).std(ddof=1)
    roll_t = roll_ic * np.sqrt(window) / roll_std

    # Plot
    fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

    ic_by_month.plot(ax=ax[0], lw=1.5, title='IC Mensual (In-Sample)', grid=True)
    ax[0].set_ylabel('Spearman IC'); ax[0].axhline(0, ls=':', c='grey')

    roll_ic.plot(ax=ax[1], color='tab:blue', lw=2, label='Rolling 12m IC (eje izq.)', grid=True)
    ax[1].set_ylabel('Rolling IC Promedio'); ax[1].axhline(0, ls=':', c='grey')

    ax_twin = ax[1].twinx()
    ax_twin.plot(roll_t, color='tab:orange', lw=1.5, ls='--', label='Rolling 12m t-stat (eje der.)')
    ax_twin.set_ylabel('Rolling t-statistic')
    ax_twin.axhline(2.0, ls='--', c='red', lw=1, label='t-stat = 2.0 (Significativo)')

    lines, labels = ax[1].get_legend_handles_labels()
    lines2, labels2 = ax_twin.get_legend_handles_labels()
    ax_twin.legend(lines + lines2, labels + labels2, loc='upper left')

    plt.tight_layout()
    plt.show()

    return pd.DataFrame({'ic': ic_by_month, 'ic_r': roll_ic, 't_r': roll_t})

# 3) Ejecutar
best_lgbm_pipeline = models_to_compare_flat['LightGBM']
ic_roll_df = rolling_ic(best_lgbm_pipeline, window=12)

# 4) Veredicto (IN-SAMPLE): criterios sencillos y claros
#    - Mediana de IC mensual > 0.03
#    - Al menos 60% de las ventanas rolling con t-stat > 2.0
if ic_roll_df.dropna().empty:
    print("Resultado: no hay suficientes datos para evaluar rolling IC/t.")
else:
    ic_median = float(ic_roll_df['ic'].median(skipna=True))
    frac_tsig = float((ic_roll_df['t_r'] > 2.0).mean())

    print(f"\nRolling IC — Mediana IC mensual: {ic_median:.4f}")
    print(f"Rolling IC — % ventanas con t-stat > 2.0: {frac_tsig:.1%}")

    ic_pass = np.isfinite(ic_median) and ic_median > 0.03
    t_pass  = np.isfinite(frac_tsig) and frac_tsig >= 0.60

    if ic_pass and t_pass:
        print("✅ PASA: dinámica in-sample saludable (IC mediano alto y t-stat consistente).")
    else:
        print("❌ FALLA: dinámica in-sample débil/sensible.")
        if not ic_pass:
            print("   - IC mediano ≤ 0.03")
        if not t_pass:
            print("   - Menos del 60% de ventanas con t-stat > 2.0")


In [ ]:
# ================================================================================
# CHEQUEO 8: Q-FACTOR TEST (OOF TEMPORAL POR MESES · WALK-FORWARD) con veredicto
# ================================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from scipy.stats import t as student_t

print("\n--- 8. Q-Factor OOF por meses (walk-forward) ---")

models_to_compare_flat = {mn: pipe
                          for fam, d in models_to_compare.items()
                          for mn, pipe in d.items()}
best_lgbm_pipeline = models_to_compare_flat['LightGBM']

def quintile_spread_oof_temporal(model, train_min=24, gap=1, hold=1, sharpe_min=0.50, t_min=2.0):
    months = pd.to_datetime(df_model['Fecha']).dt.to_period('M')
    uniq_m = np.array(sorted(pd.unique(months)))

    oof = pd.Series(np.nan, index=df_model.index, dtype=float)

    # Walk-forward OOF por mes con embargo
    for j in range(train_min + gap, len(uniq_m) - hold + 1):
        m_test_start = uniq_m[j]
        m_test_end   = uniq_m[j + hold - 1]
        m_cut        = m_test_start - gap

        tr_idx = months < m_cut
        te_idx = (months >= m_test_start) & (months <= m_test_end)

        if tr_idx.sum()==0 or te_idx.sum()==0:
            continue

        mdl = clone(model).fit(X.loc[tr_idx], y.loc[tr_idx])
        oof.loc[te_idx] = mdl.predict(X.loc[te_idx])

    # Construcción del spread mensual OOF (Q5 - Q1)
    df_q = df_model[['Fecha', TARGET_COL]].copy()
    df_q['pred'] = oof.values
    df_q = df_q.dropna()

    def spread_func(g):
        if len(g) < 5: 
            return np.nan
        r = g['pred'].rank(pct=True)
        long_ret  = g.loc[r >= 0.80, TARGET_COL].mean()
        short_ret = g.loc[r <= 0.20, TARGET_COL].mean()
        return long_ret - short_ret

    spr = (df_q.groupby(pd.to_datetime(df_q['Fecha']).dt.to_period('M'))
                .apply(spread_func)
                .dropna())

    if spr.empty:
        print("No se pudieron calcular spreads.")
        return spr

    # Curva de equity
    cum_spread = (1 + spr).cumprod()
    ax = cum_spread.plot(figsize=(10,4), lw=2, title='Equity — Quintil Long-Short (OOF temporal)')
    ax.axhline(1, ls=':', c='grey'); ax.set_ylabel("Crecimiento de 1 USD"); plt.grid(True); plt.show()

    # Métricas y veredicto
    mean_m = spr.mean()
    std_m  = spr.std(ddof=1)
    n_m    = len(spr)
    sharpe = mean_m / std_m * np.sqrt(12) if std_m > 0 else np.nan
    tstat  = mean_m / (std_m / np.sqrt(n_m)) if (std_m > 0 and n_m > 1) else np.nan

    print(f"OOF monthly spread — mean={mean_m:.4f}, std={std_m:.4f}, n={n_m}")
    print(f"Sharpe (OOF temporal): {sharpe:.2f}")
    print(f"t-stat del spread mensual: {tstat:.2f}")

    sharpe_pass = np.isfinite(sharpe) and sharpe >= sharpe_min
    tstat_pass  = np.isfinite(tstat)  and tstat  >= t_min

    if sharpe_pass and tstat_pass:
        print(f"✅ PASA: Sharpe ≥ {sharpe_min:.2f} y t-stat ≥ {t_min:.2f}.")
    else:
        print("❌ FALLA:")
        if not sharpe_pass:
            print(f"   - Sharpe ({sharpe:.2f}) < {sharpe_min:.2f}")
        if not tstat_pass:
            print(f"   - t-stat ({tstat:.2f}) < {t_min:.2f}")

    return spr

_ = quintile_spread_oof_temporal(best_lgbm_pipeline)


In [ ]:
# ================================================================================
# CHEQUEO 9: IC POR SUB-UNIVERSO (SECTOR Y TAMAÑO) — OOS temporal
# Objetivo: Verificar si la señal predictiva es robusta a través de diferentes
# segmentos del mercado o si está concentrada en un nicho.
# ================================================================================

print("\n--- 9. Analizando IC por Sector y Tamaño (OOS temporal) ---")

from sklearn.base import clone
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 0) Aplanar modelos si falta
if 'models_to_compare_flat' not in locals():
    models_to_compare_flat = {mn: pipe
                              for fam, d in models_to_compare.items()
                              for mn, pipe in d.items()}
best_lgbm_pipeline = models_to_compare_flat['LightGBM']

# 1) OOF temporal por mes (walk-forward con embargo) → y_pred_oof
months = pd.to_datetime(df_model['Fecha']).dt.to_period('M')
uniq_m = np.array(sorted(pd.unique(months)))

oof = pd.Series(np.nan, index=df_model.index, dtype=float)
TRAIN_MIN, GAP, HOLD = 24, 1, 1
for j in range(TRAIN_MIN + GAP, len(uniq_m) - HOLD + 1):
    m_test_start = uniq_m[j]
    m_test_end   = uniq_m[j + HOLD - 1]
    m_cut        = m_test_start - GAP
    tr_idx = months < m_cut
    te_idx = (months >= m_test_start) & (months <= m_test_end)
    if tr_idx.sum()==0 or te_idx.sum()==0:
        continue
    mdl = clone(best_lgbm_pipeline).fit(X.loc[tr_idx], y.loc[tr_idx])
    oof.loc[te_idx] = mdl.predict(X.loc[te_idx])

# 2) Helper para IC por bucket (OOS)
def ic_by_bucket_oos(pred_series, bucket_col):
    tmp = pd.concat([
        df.loc[df_model.index, bucket_col],
        y.rename('target'),
        pred_series.rename('pred')
    ], axis=1).dropna(subset=[bucket_col, 'pred'])
    ic_series = (tmp.groupby(bucket_col)
                   .apply(lambda g: spearmanr(g['pred'], g['target'])[0])
                   .sort_values(ascending=False))
    plt.figure(figsize=(12,5))
    ic_series.plot(kind='bar', grid=True)
    plt.title(f'Information Coefficient (OOF) por {bucket_col}')
    plt.ylabel('Spearman IC'); plt.xlabel('')
    plt.xticks(rotation=45, ha='right'); plt.axhline(0, ls=':', c='grey')
    plt.tight_layout(); plt.show()
    return ic_series

# 3) IC por Sector (OOF)
ic_sector = ic_by_bucket_oos(oof, 'Sector_GICS')
print("IC OOF por Sector:\n", ic_sector, '\n')

# 4) IC por Tamaño (OOF). Crear size_bucket si falta (usar Cap de mercado ya anclado)
if 'size_bucket' not in df.columns:
    df['size_bucket'] = pd.qcut(df['Cap de mercado'], q=3, labels=['Small','Mid','Large'])
ic_size = ic_by_bucket_oos(oof, 'size_bucket')
print("IC OOF por Tamaño:\n", ic_size)

# 5) Veredicto
IC_THRESHOLD_BUCKET = 0.10
print("\n--- Veredicto del Test por Sub-Universo (OOF) ---")
sec_ok  = (ic_sector.min() > IC_THRESHOLD_BUCKET) if len(ic_sector) else False
size_ok = (ic_size.min()   > IC_THRESHOLD_BUCKET) if len(ic_size)   else False
if sec_ok and size_ok:
    print("✅ PASA: Señal robusta en todos los sectores y tamaños analizados (OOF).")
else:
    print("❌ FALLA: Debilidades por sub-universo (OOF).")
    if not sec_ok and len(ic_sector):
        print(f"   - Peor sector: {ic_sector.idxmin()} (IC={ic_sector.min():.4f} ≤ {IC_THRESHOLD_BUCKET:.2f})")
    if not size_ok and len(ic_size):
        print(f"   - Peor tamaño: {ic_size.idxmin()} (IC={ic_size.min():.4f} ≤ {IC_THRESHOLD_BUCKET:.2f})")


#### último set de sanity-checks

In [ ]:
# ================================================================================
# CHEQUEO 10: TEST DE NEUTRALIZACIÓN POR SECTOR — OOS temporal
# Objetivo: Determinar si el poder predictivo del modelo proviene de la selección
# de acciones dentro de los sectores (alfa puro) o de las apuestas sectoriales.
# ================================================================================

print("\n--- 10. Neutralización por Sector (predicciones OOF temporales) ---")

from sklearn.base import clone
from scipy.stats import spearmanr
import numpy as np
import pandas as pd

# 1) Predicciones OOF temporales (si no quedaron del chequeo 9)
if 'oof' not in locals() or oof.isna().all():
    months = pd.to_datetime(df_model['Fecha']).dt.to_period('M')
    uniq_m = np.array(sorted(pd.unique(months)))
    oof = pd.Series(np.nan, index=df_model.index, dtype=float)
    TRAIN_MIN, GAP, HOLD = 24, 1, 1
    for j in range(TRAIN_MIN + GAP, len(uniq_m) - HOLD + 1):
        m_test_start = uniq_m[j]
        m_test_end   = uniq_m[j + HOLD - 1]
        m_cut        = m_test_start - GAP
        tr_idx = months < m_cut
        te_idx = (months >= m_test_start) & (months <= m_test_end)
        if tr_idx.sum()==0 or te_idx.sum()==0:
            continue
        mdl = clone(best_lgbm_pipeline).fit(X.loc[tr_idx], y.loc[tr_idx])
        oof.loc[te_idx] = mdl.predict(X.loc[te_idx])

# 2) Neutralizar por sector dentro de cada fecha (demean)
tmp = pd.DataFrame({
    'Fecha': df.loc[df_model.index, 'Fecha'].values,
    'Sector_GICS': df.loc[df_model.index, 'Sector_GICS'].values,
    'y_true': y.values,
    'y_pred': oof.values
}).dropna(subset=['Sector_GICS','y_pred'])

tmp['y_pred_neut'] = (tmp.groupby(['Fecha','Sector_GICS'])['y_pred']
                        .transform(lambda s: s - s.mean()))

# 3) IC OOF antes vs. después (por mes)
def _safe_ic(g):
    return spearmanr(g['y_true'], g['y_pred'])[0]
def _safe_ic_neut(g):
    return spearmanr(g['y_true'], g['y_pred_neut'])[0]

ic_m_raw  = tmp.groupby(pd.to_datetime(tmp['Fecha']).dt.to_period('M')).apply(_safe_ic).dropna()
ic_m_neut = tmp.groupby(pd.to_datetime(tmp['Fecha']).dt.to_period('M')).apply(_safe_ic_neut).dropna()

ic_raw_mean  = float(ic_m_raw.mean())  if len(ic_m_raw)  else np.nan
ic_neut_mean = float(ic_m_neut.mean()) if len(ic_m_neut) else np.nan

print("\n--- Resultados (OOF por mes) ---")
print(f"  - IC OOF (raw)  mean: {ic_raw_mean:.4f}")
print(f"  - IC OOF (neut) mean: {ic_neut_mean:.4f}")

# 4) Veredicto (decay relativo)
IC_DECAY_THRESHOLD = 0.30
if np.isfinite(ic_raw_mean) and ic_raw_mean > 0:
    decay = (ic_raw_mean - ic_neut_mean) / ic_raw_mean
else:
    decay = np.nan

print(f"  - Caída relativa del IC tras neutralizar: {decay:.2%}" if np.isfinite(decay) else "  - Caída: N/A")

if np.isfinite(decay) and decay < IC_DECAY_THRESHOLD:
    print("✅ PASA: Alfa principalmente intra-sector (stock-picking).")
else:
    print("❌ FALLA (matiz): gran parte del poder proviene de bets sectoriales.")


Fama-French

In [ ]:
# -------------------------------------------------------------------
# 1) Instalar / importar lo necesario
# -------------------------------------------------------------------
# pip install pandas-datareader  (si aún no lo tienes)
import pandas_datareader.data as web
import pandas as pd

# -------------------------------------------------------------------
# 2) Descargar los 3-Factores (mensuales) desde la web de Ken French
# -------------------------------------------------------------------
# - 'F-F_Research_Data_Factors'  → 3 factores + RF
# - El índice viene como periodo (1926-01, …); lo convertimos a Timestamp
ff_raw      = web.DataReader('F-F_Research_Data_Factors',
                             data_source='famafrench',
                             start='2000-01')[0]   # tabla 0 = mensuales
ff_monthly  = ff_raw / 100.0                      # pasa a decimal
ff_monthly.index = ff_monthly.index.to_timestamp()  # Period → datetime-últ día mes

# -------------------------------------------------------------------
# 3) Dejamos sólo las columnas que vamos a usar y renombramos
# -------------------------------------------------------------------
df_ff = ff_monthly.rename(columns={
           'Mkt-RF': 'MKT_RF',
           'SMB'   : 'SMB',
           'HML'   : 'HML',
           'RF'    : 'RF'
       })

print(df_ff.head())


In [ ]:
# ================================================================================
# CHEQUEO 11: REGRESIÓN DE FACTORES FAMA–FRENCH (FF3 / FF5 + MOM) — OOS temporal
# Objetivo: Estimar si la señal genera α genuino (no explicado por factores).
# Metodología: spread mensual OOS (walk-forward con embargo) → OLS HAC (Newey–West)
# ================================================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
from pandas_datareader import data as web
from sklearn.base import clone

print("\n--- 11. Ejecutando Regresión de Factores Fama–French (OOS temporal + HAC) ---")

# 0) Seleccionar modelo (a partir del diccionario aplanado)
if 'models_to_compare_flat' not in locals():
    models_to_compare_flat = {mn: pipe
                              for fam, d in models_to_compare.items()
                              for mn, pipe in d.items()}
best_lgbm_pipeline = models_to_compare_flat['LightGBM']

# 1) OOF temporal por meses (Q5-Q1) para construir la serie de spread
def quintile_spread_oof_temporal(model, X, y, dates, train_min=24, gap=1, hold=1, q=0.20):
    months = pd.to_datetime(dates).dt.to_period('M')
    uniq_m = np.array(sorted(pd.unique(months)))
    oof = pd.Series(np.nan, index=X.index, dtype=float)

    for j in range(train_min + gap, len(uniq_m) - hold + 1):
        m_test_start = uniq_m[j]
        m_test_end   = uniq_m[j + hold - 1]
        m_cut        = m_test_start - gap

        tr_idx = months < m_cut
        te_idx = (months >= m_test_start) & (months <= m_test_end)
        if tr_idx.sum() == 0 or te_idx.sum() == 0:
            continue

        mdl = clone(model).fit(X.loc[tr_idx], y.loc[tr_idx])
        oof.loc[te_idx] = mdl.predict(X.loc[te_idx])

    aux = pd.DataFrame({'Fecha': pd.to_datetime(dates), 'pred': oof, 'ret': y}).dropna()
    def _spr(g):
        # Q5 - Q1 (fallback a NaN si hay <5 nombres)
        r = g['pred'].rank(pct=True)
        return g.loc[r >= 1-q, 'ret'].mean() - g.loc[r <= q, 'ret'].mean() if len(g) >= 5 else np.nan

    spread_m = aux.groupby(aux['Fecha'].dt.to_period('M')).apply(_spr).dropna()
    spread_m.index = spread_m.index.to_timestamp('M')  # Period[M] → MonthEnd Timestamp
    spread_m.name = 'spread'
    return spread_m

# 2) Cargar factores FF mensuales (FF3 + opcional FF5 y MOM), en decimales
def load_ff_factors(include_ff5=True, include_mom=True):
    ff3 = web.DataReader('F-F_Research_Data_Factors', 'famafrench')[0] / 100.0
    ff3.index = ff3.index.to_timestamp('M')
    ff3 = ff3.rename(columns={'Mkt-RF':'MKT_RF', 'SMB':'SMB', 'HML':'HML', 'RF':'RF'})
    dfs = [ff3[['MKT_RF','SMB','HML','RF']]]

    if include_ff5:
        ff5 = web.DataReader('F-F_Research_Data_5_Factors_2x3', 'famafrench')[0] / 100.0
        ff5.index = ff5.index.to_timestamp('M')
        ff5 = ff5.rename(columns={'Mkt-RF':'MKT_RF'})
        dfs.append(ff5[['RMW','CMA']])

    if include_mom:
        mom = web.DataReader('F-F_Momentum_Factor', 'famafrench')[0] / 100.0
        mom.index = mom.index.to_timestamp('M')
        mom = mom.rename(columns={'Mom   ':'MOM'})
        dfs.append(mom[['MOM']])

    df_fac = pd.concat(dfs, axis=1).sort_index()
    # Asegurar columnas opcionales si faltan
    for c in ['RF','RMW','CMA','MOM']:
        if c not in df_fac.columns:
            df_fac[c] = np.nan
    return df_fac

# 3) Construir spread OOS mensual y preparar regresión
spread_m = quintile_spread_oof_temporal(
    model=best_lgbm_pipeline, X=X, y=y, dates=df_model['Fecha'],
    train_min=24, gap=1, hold=1, q=0.20
)

if spread_m.empty:
    print("↯ No se pudo construir una serie mensual de spreads OOS (insuficiencia de datos).")
else:
    factors = load_ff_factors(include_ff5=True, include_mom=True)  # FF3 + (RMW,CMA) + MOM

    # Alinear por MonthEnd y crear exceso sobre RF
    reg_df = pd.DataFrame(spread_m).join(factors, how='inner').dropna(subset=['spread'])
    if 'RF' in reg_df.columns and reg_df['RF'].notna().any():
        reg_df['excess_spread'] = reg_df['spread'] - reg_df['RF']
    else:
        reg_df['excess_spread'] = reg_df['spread']

    # Selección de factores disponibles
    factor_cols = [c for c in ['MKT_RF','SMB','HML','RMW','CMA','MOM'] if c in reg_df.columns]
    y_reg = reg_df['excess_spread']
    X_reg = sm.add_constant(reg_df[factor_cols])

    # 4) OLS con Newey–West (HAC) para corregir autocorrelación/heterocedasticidad
    ols = sm.OLS(y_reg, X_reg).fit(cov_type='HAC', cov_kwds={'maxlags': 6})

    # 5) Resultados clave + veredicto
    print("\n=== Fama–French regression (HAC/Newey–West) ===")
    print(f"Fechas: {reg_df.index.min().date()} → {reg_df.index.max().date()}  |  n={len(reg_df)} meses")
    print(f"Factores usados: {factor_cols}")
    print(ols.summary())

    alpha_m  = float(ols.params['const'])
    alpha_t  = float(ols.tvalues['const'])
    alpha_an = alpha_m * 12.0

    print("\n--- Alpha (ex-RF) ---")
    print(f"α mensual (HAC) : {alpha_m:.4%}")
    print(f"α anualizado    : {alpha_an:.2%}")
    print(f"t-stat(α, HAC)  : {alpha_t:.2f}")

    # Umbral de significancia clásica
    if np.isfinite(alpha_t) and alpha_t > 2.0:
        print("✅ PASA: α OOS estadísticamente significativo tras controlar por factores FF (y MOM si está).")
    else:
        print("❌ FALLA: α no significativo luego de controlar por los factores de riesgo.")


In [ ]:
# --- Opcional: Alpha rolling (24m) usando el reg_df del Chequeo 11 ---
# Nota: esto es rolling IN-SAMPLE sobre 'excess_spread' ~ factores (sin HAC en rolling).
# Úsalo como diagnóstico visual, no como prueba final de significancia.

def rolling_alpha_beta(reg_df, factors, win=24):
    rows = []
    for i in range(len(reg_df) - win + 1):
        sub = reg_df.iloc[i:i+win]
        mod = sm.OLS(sub['excess_spread'], sm.add_constant(sub[factors])).fit()
        row = {
            'Fecha': sub.index[-1],
            'Alpha': mod.params.get('const', float('nan')),
            'Alpha_t': mod.tvalues.get('const', float('nan'))
        }
        row.update({f: mod.params.get(f, float('nan')) for f in factors})
        rows.append(row)
    return pd.DataFrame(rows).set_index('Fecha')

# Usar las mismas variables del Chequeo 11:
# - reg_df (con 'excess_spread')
# - factor_cols (lista de factores usados en la regresión principal)
rolling = rolling_alpha_beta(reg_df, factor_cols, win=24)
ax = rolling[['Alpha']].plot(title='Alpha rolling 24m (diagnóstico in-sample)', figsize=(12,4))
ax.axhline(0, ls=':', c='grey')


In [ ]:
# ===============================================================================
# BLOQUE · Resumen PASA/FALLA de sanity checks (robusto)
#  - Evalúa contra umbrales los chequeos que existan en el entorno.
#  - Si algo no está disponible, muestra N/A (no rompe).
#  - Guarda un CSV con el resumen.
# ===============================================================================

import os
import numpy as np
import pandas as pd

# --------- Umbrales (ajustá si querés) ----------
THRESHOLDS = {
    'perm_ic_abs'     : 0.01,   # |IC permutado| < 0.01
    'lag_ic_abs'      : 0.05,   # |IC con lag +1| < 0.05
    'forward_ic_mean' : 0.03,   # IC forward-chain > 0.03
    'purged_ic_mean'  : 0.05,   # IC purged > 0.05
    'purged_sharpe'   : 0.50,   # Sharpe bloques > 0.5
    'purged_hit'      : 0.60,   # Hit-rate > 60%
    'ic_oos_mean'     : 0.05,   # IC mensual OOS > 0.05
    'ic_oos_t'        : 2.00,   # t-stat IC mensual OOS > 2
    'decile_t'        : 2.00,   # t-stat spread deciles > 2
    'neutral_decay'   : 0.30,   # caída IC tras neutralizar < 30%
    'ff_alpha_t'      : 2.00,   # t-stat alpha FF > 2
}

def _status(ok: bool) -> str:
    return "✅ PASA" if ok else "❌ FALLA"

rows = []

# -------------------------
# 0) Fallbacks / aliases
# -------------------------
# Alias para IC mensual OOS si usaste otro nombre
if 'ic_m' not in globals() and 'ic_month' in globals():
    ic_m = ic_month

# Alias para alpha de Fama–French si lo tenés suelto
if 'alpha_t' in globals() and 'ff_res' not in globals():
    try:
        ff_res = {'alpha_tstat': float(alpha_t)}
        alpha_t_stat = float(alpha_t)  # retrocompatibilidad
    except Exception:
        pass

# Helper mínimo para construir spread_m desde oos, si no existe
def _ensure_spread_monthly_from_oos(oos_df: pd.DataFrame) -> pd.Series:
    df = oos_df.copy()
    if 'Mes' not in df.columns:
        df['Mes'] = pd.to_datetime(df['Fecha']).dt.to_period('M')
    def _spread(g):
        r = g['y_pred'].rank(method='first')
        try:
            if len(g) >= 10:
                d = pd.qcut(r, 10, labels=np.arange(1,11))
                m = g.assign(dec=d.astype(int)).groupby('dec')['y_true'].mean()
                return float(m.get(10, np.nan) - m.get(1, np.nan))
            elif len(g) >= 5:
                q = pd.qcut(r, 5, labels=np.arange(1,6))
                m = g.assign(q=q.astype(int)).groupby('q')['y_true'].mean()
                return float(m.get(5, np.nan) - m.get(1, np.nan))
            else:
                return np.nan
        except ValueError:
            return np.nan
    s = df.groupby('Mes', sort=True).apply(_spread).dropna()
    s.index = s.index.to_timestamp('M')
    s.name = 'spread'
    return s.sort_index()

# ------------------------------------------------
# 1) Permutación (perm_ic: array de ICs permutados)
# ------------------------------------------------
perm_ic_mean = float('nan')
if 'perm_ic' in globals() and hasattr(perm_ic, '__len__') and len(perm_ic):
    perm_ic_mean = float(np.nanmean(perm_ic))
rows.append({
    'Chequeo' : 'Permutación (IC≈0)',
    'Valor'   : f"{perm_ic_mean:.4f}" if np.isfinite(perm_ic_mean) else "N/A",
    'Criterio': f"|IC| < {THRESHOLDS['perm_ic_abs']}",
    'Estado'  : _status(abs(perm_ic_mean) < THRESHOLDS['perm_ic_abs']) if np.isfinite(perm_ic_mean) else "N/A"
})

# --------------------------------------------------------
# 2) Lag +1 (ic_mean del test de lag adicional “purged”)
# --------------------------------------------------------
lag_ic_mean = float('nan')
if 'ic_mean' in globals() and np.isfinite(ic_mean):
    lag_ic_mean = float(ic_mean)
rows.append({
    'Chequeo' : 'Lag +1 (purged)',
    'Valor'   : f"{lag_ic_mean:.4f}" if np.isfinite(lag_ic_mean) else "N/A",
    'Criterio': f"|IC| < {THRESHOLDS['lag_ic_abs']}",
    'Estado'  : _status(abs(lag_ic_mean) < THRESHOLDS['lag_ic_abs']) if np.isfinite(lag_ic_mean) else "N/A"
})

# ----------------------------------------------
# 3) Forward-chain simple (ic_chain: lista ICs)
# ----------------------------------------------
fc_ic = float('nan')
if 'ic_chain' in globals() and hasattr(ic_chain, '__len__') and len(ic_chain):
    fc_ic = float(np.nanmean(ic_chain))
rows.append({
    'Chequeo' : 'Forward-chain IC',
    'Valor'   : f"{fc_ic:.4f}" if np.isfinite(fc_ic) else "N/A",
    'Criterio': f"IC > {THRESHOLDS['forward_ic_mean']}",
    'Estado'  : _status(fc_ic > THRESHOLDS['forward_ic_mean']) if np.isfinite(fc_ic) else "N/A"
})

# -----------------------------------------------------------------
# 4) Backtest purged (ic_mean_purged, sharpe_purged, hit_rate_purged)
# -----------------------------------------------------------------
purged_ic = purged_sh = purged_hr = float('nan')
if all(k in globals() for k in ['ic_mean_purged','sharpe_purged','hit_rate_purged']):
    purged_ic = float(ic_mean_purged)
    purged_sh = float(sharpe_purged)
    purged_hr = float(hit_rate_purged)
rows += [
    {'Chequeo':'Purged IC medio', 'Valor': f"{purged_ic:.4f}" if np.isfinite(purged_ic) else "N/A",
     'Criterio': f"> {THRESHOLDS['purged_ic_mean']}",
     'Estado'  : _status(purged_ic > THRESHOLDS['purged_ic_mean']) if np.isfinite(purged_ic) else "N/A"},
    {'Chequeo':'Purged Sharpe (bloques)', 'Valor': f"{purged_sh:.2f}" if np.isfinite(purged_sh) else "N/A",
     'Criterio': f"> {THRESHOLDS['purged_sharpe']}",
     'Estado'  : _status(purged_sh > THRESHOLDS['purged_sharpe']) if np.isfinite(purged_sh) else "N/A"},
    {'Chequeo':'Purged Hit-rate (IC>0)', 'Valor': f"{purged_hr:.2%}" if np.isfinite(purged_hr) else "N/A",
     'Criterio': f"> {THRESHOLDS['purged_hit']:.0%}",
     'Estado'  : _status(purged_hr > THRESHOLDS['purged_hit']) if np.isfinite(purged_hr) else "N/A"},
]

# -----------------------------------------------------------
# 5) IC mensual OOS y t-stat (usa ic_m del kit OOS si existe)
# -----------------------------------------------------------
ic_oos_mean = ic_oos_t = float('nan')
if 'ic_m' in globals() and hasattr(ic_m, 'values') and len(ic_m.dropna()):
    _ic_vals = ic_m.dropna().values.astype(float)
    ic_oos_mean = float(np.nanmean(_ic_vals))
    if len(_ic_vals) > 1:
        ic_oos_t = float(ic_oos_mean / (np.nanstd(_ic_vals, ddof=1) / np.sqrt(len(_ic_vals))))
rows += [
    {'Chequeo':'IC mensual OOS (mean)', 'Valor': f"{ic_oos_mean:.4f}" if np.isfinite(ic_oos_mean) else "N/A",
     'Criterio': f"> {THRESHOLDS['ic_oos_mean']}",
     'Estado'  : _status(ic_oos_mean > THRESHOLDS['ic_oos_mean']) if np.isfinite(ic_oos_mean) else "N/A"},
    {'Chequeo':'IC mensual OOS (t-stat)', 'Valor': f"{ic_oos_t:.2f}" if np.isfinite(ic_oos_t) else "N/A",
     'Criterio': f"> {THRESHOLDS['ic_oos_t']}",
     'Estado'  : _status(ic_oos_t > THRESHOLDS['ic_oos_t']) if np.isfinite(ic_oos_t) else "N/A"},
]

# ---------------------------------------------------------------------
# 6) Decile/Quintile spread OOS t-stat (usa spread_m o lo arma desde oos)
# ---------------------------------------------------------------------
dec_t = float('nan')
if 'spread_m' in globals() and hasattr(spread_m, 'values') and len(spread_m.dropna()) > 1:
    _spr_vals = spread_m.dropna().values.astype(float)
    mean_ = float(np.nanmean(_spr_vals))
    std_  = float(np.nanstd(_spr_vals, ddof=1))
    if std_ > 0:
        dec_t = float(mean_ / (std_ / np.sqrt(len(_spr_vals))))
elif 'oos' in globals() and isinstance(oos, pd.DataFrame) and not oos.empty:
    try:
        _spr = _ensure_spread_monthly_from_oos(oos)
        if len(_spr.dropna()) > 1:
            mean_ = float(_spr.mean())
            std_  = float(_spr.std(ddof=1))
            if std_ > 0:
                dec_t = float(mean_ / (std_ / np.sqrt(len(_spr))))
    except Exception:
        pass
rows.append({
    'Chequeo' : 'Decile spread OOS (t-stat)',
    'Valor'   : f"{dec_t:.2f}" if np.isfinite(dec_t) else "N/A",
    'Criterio': f"> {THRESHOLDS['decile_t']}",
    'Estado'  : _status(dec_t > THRESHOLDS['decile_t']) if np.isfinite(dec_t) else "N/A"
})

# ----------------------------------------------------
# 7) Neutralización por sector (caída de IC relativo)
# ----------------------------------------------------
neut_decay = float('nan')
if all(k in globals() for k in ['ic_raw','ic_neut']) and (ic_raw is not None) and np.isfinite(ic_raw) and ic_raw > 0:
    try:
        neut_decay = float((ic_raw - ic_neut) / ic_raw)
    except Exception:
        pass
rows.append({
    'Chequeo' : 'Neutralización Sector (caída IC)',
    'Valor'   : f"{neut_decay:.2%}" if np.isfinite(neut_decay) else "N/A",
    'Criterio': f"< {THRESHOLDS['neutral_decay']:.0%}",
    'Estado'  : _status(neut_decay < THRESHOLDS['neutral_decay']) if np.isfinite(neut_decay) else "N/A"
})

# --------------------------------------------
# 8) Fama–French α t-stat (Cheq. 11 / ff_res)
# --------------------------------------------
alpha_t_val = float('nan')
if 'ff_res' in globals() and isinstance(ff_res, dict) and 'alpha_tstat' in ff_res:
    alpha_t_val = float(ff_res['alpha_tstat'])
elif 'alpha_t_stat' in globals() and np.isfinite(alpha_t_stat):
    alpha_t_val = float(alpha_t_stat)
elif 'alpha_t' in globals() and np.isfinite(alpha_t):
    alpha_t_val = float(alpha_t)
rows.append({
    'Chequeo' : 'Fama–French α (t-stat)',
    'Valor'   : f"{alpha_t_val:.2f}" if np.isfinite(alpha_t_val) else "N/A",
    'Criterio': f"> {THRESHOLDS['ff_alpha_t']}",
    'Estado'  : _status(alpha_t_val > THRESHOLDS['ff_alpha_t']) if np.isfinite(alpha_t_val) else "N/A"
})

# -------------------------
# Render + guardado
# -------------------------
summary_checks = pd.DataFrame(rows, columns=['Chequeo','Valor','Criterio','Estado'])
print("\n================  RESUMEN SANITY CHECKS  ================")
try:
    display(summary_checks)
except Exception:
    print(summary_checks.to_string(index=False))
print("=========================================================\n")

os.makedirs("artifacts", exist_ok=True)
summary_checks.to_csv("artifacts/sanity_checks_summary.csv", index=False, encoding="utf-8")
print("📄 Guardado en artifacts/sanity_checks_summary.csv")


In [ ]:
# ================================================================================
# BLOQUE: ANÁLISIS DE FEATURE IMPORTANCE DE LOS MEJORES MODELOS
# ================================================================================

print("\n" + "="*60)
print("Análisis de Importancia de Features de los Mejores Modelos")
print("="*60)

# --- 1. Aplanar el diccionario de modelos si aún no existe ---
if 'models_to_compare_flat' not in locals():
    models_to_compare_flat = {}
    for family, model_dict in models_to_compare.items():
        for model_name, pipeline in model_dict.items():
            models_to_compare_flat[model_name] = pipeline

# --- 2. Definir la función auxiliar ---
def plot_feature_importances(model, feature_names, title="Importancia de Características"):
    try:
        if hasattr(model, 'steps'):
            final_estimator_name = model.steps[-1][0]
            estimator = model.named_steps[final_estimator_name]
        else:
            estimator = model

        if not hasattr(estimator, 'feature_importances_'):
            print(f"Advertencia: El estimador {type(estimator).__name__} no tiene 'feature_importances_'.")
            return None
        
        importances = estimator.feature_importances_
        fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
        fi_df.sort_values('importance', ascending=False, inplace=True)
        
        top_n = 20
        fi_df = fi_df.head(top_n)

        plt.figure(figsize=(10, 8))
        plt.barh(fi_df['feature'], fi_df['importance'])
        plt.xlabel('Importancia')
        plt.ylabel('Característica')
        plt.title(title)
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()
        return fi_df
    except Exception as e:
        print(f"Error al graficar importancia para {title}: {e}")
        return None

# --- 3. Seleccionar los 3 Mejores Modelos del Leaderboard ---
try:
    if 'df_leaderboard' not in locals() or df_leaderboard.empty:
        raise NameError("'df_leaderboard' no está definido o está vacío.")
    
    top_3_configs = df_leaderboard.sort_values(by='Spearman', ascending=False).head(3)
    print("\n--- Top 3 Modelos Seleccionados para Análisis (por Spearman) ---")
    display(top_3_configs[['Familia', 'Modelo', 'RMSE', 'R2', 'Spearman']])

except (NameError, KeyError) as e:
    print(f"Error al seleccionar los mejores modelos: {e}")
    top_3_configs = pd.DataFrame()

# --- 4. Re-entrenar y Graficar Importancia para Cada Modelo ---
if not top_3_configs.empty:
    # Verificar que las variables necesarias existan
    if 'X' not in locals() or 'y' not in locals():
        print("ERROR: Los DataFrames 'X' y/o 'y' no están definidos.")
    elif 'predictor_cols_final' not in locals():
        print("ERROR: La lista 'predictor_cols_final' no está definida.")
    else:
        for index, config_row in top_3_configs.iterrows():
            model_name = config_row['Modelo']
            # Obtener el pipeline base del diccionario aplanado
            pipeline_to_train = models_to_compare_flat.get(model_name)

            if pipeline_to_train is None:
                print(f"\nNo se encontró el pipeline para '{model_name}'. Saltando.")
                continue

            print("\n" + "-"*50)
            print(f"Re-entrenando y analizando: {model_name}")
            print("-"*50)

            try:
                start_fit_time = time.time()
                pipeline_to_train.fit(X, y)
                end_fit_time = time.time()
                print(f"  Modelo re-entrenado en el dataset completo en {end_fit_time - start_fit_time:.2f} seg.")

                # Graficar importancia de features usando el nombre de variable correcto
                plot_feature_importances(
                    model=pipeline_to_train,
                    feature_names=predictor_cols_final, # <-- CORRECCIÓN APLICADA AQUÍ
                    title=f"Importancia de Features - {model_name}"
                )

            except Exception as e:
                print(f"  ERROR al re-entrenar o graficar para {model_name}: {e}")
else:
    print("\nNo se seleccionaron modelos para analizar (leaderboard vacío).")

In [ ]:
# ================================================================================
# BLOQUE: ANÁLISIS DE INTERPRETABILIDAD CON SHAP
# Objetivo: Entender no solo QUÉ features son importantes, sino CÓMO afectan
# las predicciones del modelo.
# ================================================================================
import shap

print("\n" + "="*80)
print("### INICIO: Análisis de Interpretabilidad con SHAP ###")
print("="*80)

# --- ASUNCIONES ---
# - best_trained_models: Diccionario con los mejores modelos ya entrenados.
# - X: El DataFrame completo de features.
# - predictor_cols_final: La lista de nombres de las features.

# --- 1. Seleccionar el mejor modelo para el análisis ---
# Usaremos XGBoost, que fue uno de los más consistentes.
try:
    best_model_name = 'XGBoost' # O el que prefieras de tus mejores modelos
    # El modelo ya está dentro de un pipeline, necesitamos extraer el estimador final
    pipeline_fitted = best_trained_models[best_model_name]
    
    # Extraer el modelo de árbol y el escalador (si existe) del pipeline
    if 'scaler' in pipeline_fitted.named_steps:
        scaler = pipeline_fitted.named_steps['scaler']
        model_fitted = pipeline_fitted.named_steps['model']
        # Transformar los datos con el escalador ya ajustado
        X_transformed = scaler.transform(X)
        X_transformed_df = pd.DataFrame(X_transformed, columns=X.columns, index=X.index)
    else:
        model_fitted = pipeline_fitted.named_steps['model']
        X_transformed_df = X # No hay escalador, usar X directamente

    print(f"Analizando el modelo: {best_model_name}")

except (NameError, KeyError) as e:
    print(f"Error: No se pudo encontrar el modelo entrenado o los datos. {e}")
    model_fitted = None

# --- 2. Calcular los valores SHAP ---
if model_fitted:
    print("   Calculando los valores SHAP... (Esto puede tardar unos minutos)")
    # Usamos TreeExplainer, que es muy eficiente para modelos de árbol
    explainer = shap.TreeExplainer(model_fitted)
    shap_values = explainer.shap_values(X_transformed_df)
    print("   Cálculo de SHAP completado.")

    # --- 3. Visualizar los resultados de SHAP ---
    
    # a) Gráfico de Resumen (Summary Plot) - El más importante
    # Muestra la importancia global y el impacto de cada feature.
    print("\n--- Gráfico de Resumen SHAP (Importancia Global) ---")
    shap.summary_plot(
        shap_values,
        X_transformed_df,
        plot_type="bar",
        show=False
    )
    plt.title(f"Importancia Media de Features (SHAP) - {best_model_name}")
    plt.show()

    # b) Gráfico de Densidad (Beeswarm Plot) - El más informativo
    # Muestra el impacto de cada feature para cada observación.
    # - Eje X: Valor SHAP (impacto en la predicción)
    # - Color: Valor de la feature (rojo=alto, azul=bajo)
    print("\n--- Gráfico de Densidad SHAP (Impacto y Dirección) ---")
    shap.summary_plot(
        shap_values,
        X_transformed_df,
        show=False
    )
    plt.title(f"Impacto de Features en las Predicciones (SHAP) - {best_model_name}")
    plt.show()

    # c) Gráfico de Dependencia (Dependence Plot) - Para una feature específica
    # Muestra cómo el impacto de una feature cambia según su valor.
    # Vamos a analizar la feature más importante (asumiendo que es una de las de sorpresa macro)
    try:
        # Obtener el nombre de la feature más importante del summary plot
        shap_sum = np.abs(shap_values).mean(axis=0)
        most_important_feature = X_transformed_df.columns[np.argmax(shap_sum)]
        
        print(f"\n--- Gráfico de Dependencia SHAP para la feature más importante: '{most_important_feature}' ---")
        shap.dependence_plot(
            most_important_feature,
            shap_values,
            X_transformed_df,
            interaction_index="auto", # Colorea por la feature que más interactúa
            show=False
        )
        plt.title(f"Impacto de '{most_important_feature}' en las Predicciones")
        plt.show()
    except Exception as e:
        print(f"No se pudo generar el gráfico de dependencia: {e}")